<div style="border-left:6px solid #b30338; padding:8px 16px;">

# MCDI505 — Análisis de Series de Tiempo
## Sesión 2 · Métodos clásicos: la metodología Box–Jenkins (ARIMA / SARIMA)

**Magíster en Ciencia de Datos · UNAB Online**  
**Duración estimada:** 90 min de trabajo guiado + 60 min de laboratorio autónomo

</div>

---

En la Sesión 1 aprendimos a **diagnosticar** una serie: reconocer sus componentes, medir su dependencia con la ACF y decidir si es estacionaria. Esta sesión responde a la pregunta que quedó abierta: **¿cómo se construye un modelo sobre esa dependencia y cómo se prueba que el modelo es adecuado?**

La respuesta es la **metodología Box–Jenkins**, el procedimiento iterativo de identificación, estimación y verificación que sigue siendo el estándar profesional para series univariadas y la referencia obligada contra la cual se comparan los métodos modernos.

### Requisitos previos
Contenidos de la Sesión 1: ACF y PACF con sus bandas, estacionariedad débil, pruebas ADF y KPSS, diferenciación regular y estacional, y el criterio de varianza para detectar sobre-diferenciación.

### Cómo trabajar este notebook
1. **Ejecute las celdas en orden.** El notebook es **autocontenido**: no requiere descargar archivos ni conexión a internet. La serie de trabajo está incrustada en el código.
2. Las celdas marcadas con 🧭 **Pausa analítica** contienen preguntas que debe responder *antes* de continuar. Escriba su respuesta en la celda de texto siguiente.
3. Cada decisión técnica se documenta en el formato **decisión / evidencia / alternativa descartada / consecuencia**, el mismo que exige el caso transversal.
4. El §13 (Laboratorio) constituye el **entregable de la Evaluación Formativa 2** y se desarrolla sobre la serie del proyecto, no sobre la serie de esta sesión.

### Reproducibilidad
Semilla global `SEMILLA = 505`. La estimación por máxima verosimilitud es determinista dada la serie y el modelo, de modo que sus resultados numéricos deben coincidir con los del enunciado hasta la precisión del optimizador.

---
# 0. El problema que guía la sesión

> ### 📌 Caso: *Compromiso de capacidad en una aerolínea*
>
> Usted trabaja en el área de Planificación de Flota de una aerolínea internacional. En diciembre, la compañía debe comprometer con los arrendadores la **capacidad que operará durante los doce meses siguientes**: cuántas aeronaves mantiene en contrato y con qué distribución mensual de asientos ofertados.
>
> El compromiso es vinculante y el error tiene dos costos distintos:
>
> - **Comprometer de menos** obliga a arrendar capacidad de emergencia en temporada alta, a tarifas que multiplican varias veces el costo contractual, y a dejar demanda sin atender.
> - **Comprometer de más** implica pagar aeronaves estacionadas.
>
> La Gerencia le entrega el registro mensual de pasajeros transportados de los últimos años y le hace tres preguntas:
>
> 1. ¿Cuántos pasajeros esperamos mes a mes el próximo año?
> 2. ¿Con qué incertidumbre? *No queremos un número, queremos un rango.*
> 3. ¿Por qué deberíamos creerle a ese modelo y no al procedimiento que usamos hoy, que consiste en tomar el mismo mes del año pasado y agregarle el crecimiento promedio?

La tercera pregunta es la más importante de la sesión, y es la que ordena todo lo que sigue. Un modelo no se justifica porque sea sofisticado: se justifica porque **supera de forma medible a la alternativa trivial** y porque **sus supuestos se verifican**. Ambas cosas hay que demostrarlas, y esta sesión trata precisamente de cómo se demuestran.

> ### 🔗 Relación con el caso transversal
> La serie de esta sesión (pasajeros de aerolíneas) es el **caso guía**: sobre ella se desarrolla la técnica paso a paso. El **entregable evaluado** (§13) se desarrolla sobre la serie del **caso transversal** —consumo eléctrico del segmento comercial de DEL—, que presenta dificultades adicionales que aquí no aparecen: valores faltantes, un dato atípico y un quiebre exógeno prolongado. Traspasar el método de una serie limpia a una serie sucia es exactamente lo que se evalúa.

---
# 1. Configuración del entorno

In [ ]:
# --- Bibliotecas ---
import sys
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy
from scipy import stats
import statsmodels
import statsmodels.api as sm
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.stats.diagnostic import acorr_ljungbox, het_arch, het_goldfeldquandt
from statsmodels.tsa.stattools import acf, pacf, adfuller, kpss
from statsmodels.tsa.statespace.sarimax import SARIMAX

warnings.filterwarnings("ignore")   # silenciamos avisos de interpolacion de p-valores

# --- Reproducibilidad ---
SEMILLA = 505
rng = np.random.default_rng(SEMILLA)

# --- Estetica comun de los graficos (identica a la Sesion 1) ---
ROJO, AZUL, VERDE = "#b30338", "#0057b8", "#2e7d32"
NARANJA, MORADO, GRIS = "#ef6c00", "#6a1b9a", "#455a64"

plt.rcParams.update({
    "figure.figsize": (11, 3.6),
    "figure.dpi": 110,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.titlesize": 11,
    "font.size": 9,
    "lines.linewidth": 1.2,
})

print(f"Python       {sys.version.split()[0]}")
print(f"numpy        {np.__version__}")
print(f"pandas       {pd.__version__}")
print(f"scipy        {scipy.__version__}")
print(f"statsmodels  {statsmodels.__version__}")
print("\nEntorno listo. Semilla global:", SEMILLA)

---
# 2. La serie del caso

La serie es el registro mensual de **pasajeros transportados en vuelos internacionales entre enero de 1949 y diciembre de 1960** (144 observaciones, en miles). Es la serie con la que Box y Jenkins ilustraron su metodología, razón por la cual el modelo que terminaremos seleccionando se conoce en la literatura como **modelo de aerolínea** (*airline model*).

> ### 📎 Por qué los datos van incrustados en el código
> El notebook debe ejecutarse completo en cualquier equipo, sin red y sin archivos adjuntos. Descargar la serie desde una URL introduce un punto de falla que no aporta nada al aprendizaje: si el repositorio remoto cambia de nombre o el aula no tiene salida a internet, el notebook deja de ser reproducible. La regla vale también para su entrega: **rutas relativas y datos verificables, nunca una descarga silenciosa**.

In [ ]:
# ---------------------------------------------------------------------------
# Serie de pasajeros internacionales, mensual, ene-1949 a dic-1960 (en miles).
# Fuente original: Box, Jenkins, Reinsel y Ljung (2016), Serie G.
# ---------------------------------------------------------------------------
PASAJEROS = [
    112, 118, 132, 129, 121, 135, 148, 148, 136, 119, 104, 118,   # 1949
    115, 126, 141, 135, 125, 149, 170, 170, 158, 133, 114, 140,   # 1950
    145, 150, 178, 163, 172, 178, 199, 199, 184, 162, 146, 166,   # 1951
    171, 180, 193, 181, 183, 218, 230, 242, 209, 191, 172, 194,   # 1952
    196, 196, 236, 235, 229, 243, 264, 272, 237, 211, 180, 201,   # 1953
    204, 188, 235, 227, 234, 264, 302, 293, 259, 229, 203, 229,   # 1954
    242, 233, 267, 269, 270, 315, 364, 347, 312, 274, 237, 278,   # 1955
    284, 277, 317, 313, 318, 374, 413, 405, 355, 306, 271, 306,   # 1956
    315, 301, 356, 348, 355, 422, 465, 467, 404, 347, 305, 336,   # 1957
    340, 318, 362, 348, 363, 435, 491, 505, 404, 359, 310, 337,   # 1958
    360, 342, 406, 396, 420, 472, 548, 559, 463, 407, 362, 405,   # 1959
    417, 391, 419, 461, 472, 535, 622, 606, 508, 461, 390, 432,   # 1960
]

S = 12                                     # periodo estacional: datos mensuales
indice = pd.date_range("1949-01-01", periods=len(PASAJEROS), freq="MS")
y = pd.Series(np.asarray(PASAJEROS, dtype=float), index=indice, name="pasajeros")

# --- Verificacion de integridad: si alguien edita la lista, el assert lo detecta ---
assert len(y) == 144 and y.sum() == 40363, "La serie fue alterada: revise los datos"
assert y.index.freqstr in ("MS", "M"), "El indice debe tener frecuencia mensual declarada"
print("Serie cargada y verificada ✔")
print(f"n = {len(y)} observaciones  |  {y.index.min():%b-%Y} a {y.index.max():%b-%Y}  |  frecuencia: {y.index.freqstr}")
print(f"Faltantes: {y.isna().sum()}   Mínimo: {y.min():.0f}   Máximo: {y.max():.0f}   Media: {y.mean():.1f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12.5, 3.8), gridspec_kw={"width_ratios": [2, 1]})

# --- Izquierda: la serie completa ---
axes[0].plot(y.index, y.to_numpy(), color=ROJO, lw=1.4)
axes[0].set_title("Pasajeros internacionales, mensual 1949-1960", loc="left")
axes[0].set_ylabel("miles de pasajeros")

# --- Derecha: superposicion anual (perfil estacional) ---
for anio, grupo in y.groupby(y.index.year):
    axes[1].plot(range(1, 13), grupo.to_numpy(), lw=1.0, alpha=0.85,
                 color=plt.cm.viridis((anio - 1949) / 11))
axes[1].set_title("Superposición anual: cada línea es un año", loc="left")
axes[1].set_xlabel("mes"); axes[1].set_xticks(range(1, 13))

plt.tight_layout(); plt.show()

print("Lectura del panel derecho: el PERFIL de los doce meses se repite casi idéntico")
print("año a año (máximo en julio-agosto, mínimo en noviembre), pero la AMPLITUD de ese")
print("perfil crece con el nivel de la serie. Esa combinación —forma estable, amplitud")
print("creciente— es la firma de una estacionalidad MULTIPLICATIVA.")

---
# 3. El ciclo de Box–Jenkins

La metodología no es una receta lineal: es un **ciclo** que se recorre tantas veces como sea necesario hasta que los residuos dejen de contener estructura.

```
        ┌──────────────────────────────────────────────────────────┐
        │  1. IDENTIFICACIÓN                                       │
        │     ¿Es estacionaria? ¿Qué transformación y qué d, D?    │  §4
        │     ¿Qué órdenes p, q, P, Q sugieren la ACF y la PACF?   │  §5
        └────────────────────────────┬─────────────────────────────┘
                                     ▼
        ┌──────────────────────────────────────────────────────────┐
        │  2. ESTIMACIÓN                                           │
        │     Máxima verosimilitud sobre el conjunto de            │  §5
        │     entrenamiento. Varios candidatos, no uno solo.       │
        └────────────────────────────┬─────────────────────────────┘
                                     ▼
        ┌──────────────────────────────────────────────────────────┐
        │  3. VERIFICACIÓN                                         │
        │     ¿Los residuos son ruido blanco?                      │  §7
        │     ¿El modelo supera a la referencia trivial?           │  §6
        └────────────┬──────────────────────────────┬──────────────┘
                     │ NO                           │ SÍ
                     ▼                              ▼
             volver al paso 1                 4. USO: pronóstico
             con la estructura                    con intervalos      §8
             que quedó en los residuos
```

> ### ⚠️ El error más caro de la unidad
> Ejecutar las cuatro etapas y reportar solo la última. Lo que da valor al análisis es **la fundamentación de cada decisión**, no el modelo final. Un `auto_arima` que entrega el mejor AIC sin una sola línea de justificación vale menos que un modelo peor cuya elección está argumentada con evidencia.

---
# 4. Análisis de estacionariedad y transformaciones

### 4.1 Por qué esta sección viene primero

Todo el aparato ARMA está definido **solo para procesos estacionarios**. La ACF y la PACF que usaremos en §5 para elegir órdenes son estimadores válidos únicamente bajo estacionariedad; aplicadas a una serie con tendencia miden la tendencia y no la dinámica. Por eso el paso 1 del ciclo no es "mirar la ACF", sino **llevar la serie a un terreno donde la ACF signifique algo**.

Recordemos la definición de la Sesión 1. Un proceso $\{X_t\}$ es **estacionario en covarianza** si:

1. $\mathbb{E}[X_t] = \mu$ constante;
2. $\operatorname{Var}(X_t) = \sigma^2 < \infty$ constante;
3. $\operatorname{Cov}(X_t, X_{t+k}) = \gamma_k$ depende solo del rezago $k$.

Nuestra serie viola las tres condiciones a simple vista. El trabajo consiste en encontrar la transformación **mínima** que las restablezca: mínima, porque cada operación que aplicamos destruye información.

### 4.2 El operador de rezago

Toda la notación de la unidad se apoya en el **operador de rezago** $B$, definido por $B X_t = X_{t-1}$, y por extensión $B^k X_t = X_{t-k}$. Con él:

$$\nabla X_t = X_t - X_{t-1} = (1-B)X_t \qquad \text{(diferencia regular)}$$
$$\nabla_s X_t = X_t - X_{t-s} = (1-B^s)X_t \qquad \text{(diferencia estacional)}$$

> ### ⚠️ Confusión frecuente
> $\nabla_{12}$ **no** es $\nabla$ aplicada doce veces. $(1-B^{12})$ compara cada mes con el mismo mes del año anterior; $(1-B)^{12}$ es un polinomio de grado doce que no tiene ninguna interpretación estacional. En código: `x.diff(12)` es lo correcto; `np.diff(x, n=12)` es el error.

In [ ]:
# Demostracion numerica de la advertencia anterior
muestra = y.iloc[:20]
correcto = muestra.diff(12).dropna()
incorrecto = np.diff(muestra.to_numpy(), n=12)

print("Serie original (primeros 14 valores):")
print(muestra.head(14).to_numpy())
print(f"\nx.diff(12)      -> diferencia ESTACIONAL, {len(correcto)} valores: {correcto.to_numpy()}")
print(f"np.diff(x, n=12)-> primera diferencia aplicada 12 VECES, {len(incorrecto)} valores: {np.round(incorrecto, 1)}")
print("\nNo son la misma operación y no significan lo mismo. Verifíquelo siempre.")

### 4.3 Diagnóstico visual: ¿estacionalidad aditiva o multiplicativa?

La distinción determina si necesitamos una transformación estabilizadora antes de diferenciar.

- **Aditiva:** $X_t = T_t + S_t + \varepsilon_t$. La amplitud estacional es constante; el logaritmo no aporta.
- **Multiplicativa:** $X_t = T_t \cdot S_t \cdot \varepsilon_t$. La amplitud crece proporcionalmente al nivel. Tomando logaritmos, $\log X_t = \log T_t + \log S_t + \log \varepsilon_t$: **el logaritmo convierte una estructura multiplicativa en aditiva**, que es la que los modelos ARIMA saben tratar.

La evidencia decisiva no es el gráfico de la serie, sino el **coeficiente de variación por año**: si la desviación estándar crece a la par de la media, su cociente se mantiene aproximadamente constante y la estructura es multiplicativa.

In [ ]:
def panel_estabilidad(serie, ventana=12, titulo="", color=ROJO):
    '''Media y desviacion movil: primer diagnostico visual de estacionariedad.'''
    fig, ax = plt.subplots(figsize=(11, 3.2))
    ax.plot(serie.index, serie.to_numpy(), color="#cccccc", lw=0.9, label="serie")
    ax.plot(serie.index, serie.rolling(ventana).mean().to_numpy(), color=color, lw=1.9,
            label=f"media móvil ({ventana})")
    ax.plot(serie.index, serie.rolling(ventana).std().to_numpy(), color=AZUL, lw=1.9,
            label=f"desv. estándar móvil ({ventana})")
    ax.set_title(titulo or "Estabilidad de media y varianza", loc="left")
    ax.legend(fontsize=8, ncol=3)
    plt.tight_layout(); plt.show()


panel_estabilidad(y, S, "Serie original: la media SUBE y la dispersión sube con ella")

# --- Evidencia cuantitativa: media, desviacion y coeficiente de variacion por año ---
por_anio = y.groupby(y.index.year).agg(media="mean", desv="std")
por_anio["cv_%"] = 100 * por_anio["desv"] / por_anio["media"]

print("Resumen anual")
print(por_anio.round(2).to_string())
print(f"\nLa media anual se multiplica por {por_anio['media'].iloc[-1] / por_anio['media'].iloc[0]:.2f} entre 1949 y 1960;")
print(f"la desviación estándar anual, por {por_anio['desv'].iloc[-1] / por_anio['desv'].iloc[0]:.2f}.")
print(f"El coeficiente de variación, en cambio, se mueve entre {por_anio['cv_%'].min():.1f}% y {por_anio['cv_%'].max():.1f}%:")
print("permanece en el mismo orden de magnitud. La dispersión es PROPORCIONAL al nivel")
print("=> estacionalidad MULTIPLICATIVA => corresponde una transformación estabilizadora.")

### 4.4 Qué transformación: logaritmo o Box–Cox

La familia de **Box–Cox** generaliza el logaritmo con un parámetro $\lambda$:

$$
X_t^{(\lambda)} =
\begin{cases}
\dfrac{X_t^{\lambda} - 1}{\lambda}, & \lambda \neq 0 \\[6pt]
\log X_t, & \lambda = 0
\end{cases}
$$

El caso $\lambda = 0$ es el logaritmo, $\lambda = 1$ equivale a no transformar (salvo desplazamiento) y $\lambda = 0{,}5$ es la raíz cuadrada. Estimamos $\lambda$ por máxima verosimilitud y decidimos **comparando el valor estimado con la simplicidad de la alternativa**: si $\hat{\lambda}$ está cerca de 0, el logaritmo es preferible porque sus coeficientes se leen directamente como **variaciones porcentuales**, lo que importa cuando el resultado debe explicarse a una gerencia.

In [ ]:
lam_mle = stats.boxcox_normmax(y.to_numpy(), method="mle")
print(f"λ estimado por máxima verosimilitud: {lam_mle:.4f}")

# Comparacion de la estabilidad lograda por cada opcion:
# medimos la dispersion del coeficiente de variacion anual (cuanto menor, mas estable)
candidatas = {
    "sin transformar (λ=1)": y,
    "raíz cuadrada (λ=0.5)": np.sqrt(y),
    f"Box-Cox (λ={lam_mle:.3f})": pd.Series(stats.boxcox(y.to_numpy(), lam_mle), index=y.index),
    "logaritmo (λ=0)": np.log(y),
}

filas = []
for nombre, serie in candidatas.items():
    g = serie.groupby(serie.index.year)
    cv = 100 * g.std() / g.mean().abs()
    filas.append({"transformación": nombre, "CV anual medio (%)": cv.mean(),
                  "desv. del CV anual": cv.std()})
tabla_bc = pd.DataFrame(filas).set_index("transformación")
print("\nEstabilidad de la dispersión relativa (menor desviación del CV = varianza más estable)")
print(tabla_bc.round(3).to_string())

fig, axes = plt.subplots(1, 2, figsize=(12.5, 3.4))
axes[0].plot(y.index, y.to_numpy(), color=ROJO, lw=1.2)
axes[0].set_title("Original: amplitud estacional creciente", loc="left")
axes[1].plot(y.index, np.log(y).to_numpy(), color=MORADO, lw=1.2)
axes[1].set_title("log(x): amplitud estacional aproximadamente constante", loc="left")
plt.tight_layout(); plt.show()

> ### 📋 Decisión 1 — Transformación estabilizadora
>
> | Elemento | Contenido |
> |---|---|
> | **Decisión** | Aplicar logaritmo natural, $Z_t = \log X_t$. |
> | **Evidencia** | El coeficiente de variación anual se mantiene estable mientras media y desviación se multiplican por factores comparables (§4.3); $\hat{\lambda}_{\text{Box–Cox}} \approx 0{,}15$, cercano a 0. |
> | **Alternativa descartada** | Box–Cox con $\hat{\lambda}$ estimado. Estabiliza marginalmente mejor, pero introduce un parámetro adicional estimado de los datos, complica la retransformación de los intervalos y sus coeficientes no son interpretables. |
> | **Consecuencia** | Todo el modelamiento ocurre en escala logarítmica. Los pronósticos deberán **retransformarse** con la corrección de sesgo del §6.4 antes de reportarse en pasajeros. |
>
> Retenga el matiz: $\hat{\lambda} = 0{,}15$ no es exactamente 0. Volveremos sobre esta pequeña diferencia en §7, cuando el diagnóstico de residuos la haga visible.

### 4.5 Pruebas formales de raíz unitaria

Dos pruebas con **hipótesis nulas opuestas**, que por eso se usan en conjunto:

| Prueba | $H_0$ | $H_1$ | Se rechaza $H_0$ cuando… |
|---|---|---|---|
| **ADF** (Dickey–Fuller aumentada) | Existe raíz unitaria → **no** estacionaria | Estacionaria | $p < 0{,}05$ ⇒ evidencia **a favor** de estacionariedad |
| **KPSS** (Kwiatkowski *et al.*) | Estacionaria (en nivel o tendencia) | Existe raíz unitaria | $p < 0{,}05$ ⇒ evidencia **en contra** de estacionariedad |

### Matriz de decisión ADF × KPSS

| | **KPSS no rechaza** | **KPSS rechaza** |
|---|---|---|
| **ADF rechaza** | ✅ Estacionaria. Proceda a identificar. | ⚠️ Estacionaria en tendencia: elimine una tendencia determinista. |
| **ADF no rechaza** | ⚠️ Poca información o memoria larga: revise, considere ARFIMA. | 🔁 No estacionaria: **diferencie** y vuelva a evaluar. |

> ### ⚠️ Sobre los $p$-valores que reportan estas pruebas
> `statsmodels` interpola los $p$-valores de KPSS en una tabla acotada al intervalo $[0{,}01;\ 0{,}10]$. Un `p = 0.10` significa "**mayor o igual a** 0,10", y un `p = 0.01` significa "**menor o igual a** 0,01". Reportarlos como si fueran valores exactos es un error de interpretación frecuente. La advertencia que emite la librería es precisamente esa, y por eso la silenciamos solo después de entenderla.

Ninguna de las dos pruebas es concluyente por sí sola: ambas tienen **baja potencia** en muestras cortas, y 144 observaciones lo son. La decisión final combina las pruebas con el gráfico, la ACF y el criterio de varianza.

In [ ]:
def diagnostico_estacionariedad(serie, nombre="serie", alpha=0.05, imprimir=True):
    '''Aplica ADF y KPSS y entrega una conclusion conjunta.

    Devuelve un dict con los p-valores, la varianza y el veredicto combinado.
    Es la misma funcion de la Sesion 1, ampliada con el criterio de varianza.
    '''
    x = np.asarray(serie, dtype=float)
    x = x[~np.isnan(x)]
    p_adf = adfuller(x, autolag="AIC")[1]
    p_kpss = kpss(x, regression="c", nlags="auto")[1]
    adf_est = p_adf < alpha        # ADF:  H0 = raiz unitaria (NO estacionaria)
    kpss_est = p_kpss > alpha      # KPSS: H0 = estacionaria

    if adf_est and kpss_est:
        veredicto = "ESTACIONARIA (ambas pruebas concuerdan)"
    elif not adf_est and not kpss_est:
        veredicto = "NO ESTACIONARIA (ambas concuerdan) -> diferenciar"
    elif adf_est and not kpss_est:
        veredicto = "AMBIGUO: posible estacionariedad en tendencia"
    else:
        veredicto = "AMBIGUO: ADF no concluye; revise con ACF y varianza"

    salida = {"serie": nombre, "p_adf": p_adf, "p_kpss": p_kpss,
              "var": float(np.var(x)), "rho1": float(acf(x, nlags=1)[1]),
              "n": len(x), "veredicto": veredicto}
    if imprimir:
        print(f"── {nombre} " + "─" * max(0, 52 - len(nombre)))
        print(f"   ADF  p = {p_adf:7.4f}  (H0: raíz unitaria)  → {'estacionaria' if adf_est else 'NO estacionaria'}")
        print(f"   KPSS p = {p_kpss:7.4f}  (H0: estacionaria)   → {'estacionaria' if kpss_est else 'NO estacionaria'}")
        print(f"   VEREDICTO: {veredicto}\n")
    return salida


_ = diagnostico_estacionariedad(y, "Serie original")
_ = diagnostico_estacionariedad(np.log(y), "log(serie)")

Note el resultado del logaritmo: **el logaritmo estabiliza la varianza pero no elimina la tendencia**. Ambas pruebas siguen indicando no estacionariedad, y así debe ser: el logaritmo es una transformación monótona que no toca el nivel. Para la media hace falta diferenciar.

### 4.6 La escalera de diferenciación y el criterio de varianza

El orden importa. Aplicamos primero la diferencia **estacional** —que es la que remueve el componente dominante en una serie mensual— y luego evaluamos si aún queda deriva. En cada peldaño registramos tres señales:

1. Los $p$-valores de ADF y KPSS;
2. la **varianza** de la serie resultante;
3. la autocorrelación en el rezago 1, $\rho_1$.

> ### 🎯 Criterio de varianza (Sesión 1, §7)
> **Diferencie mientras la varianza baje; deténgase cuando empiece a subir.** Una diferencia de más introduce autocorrelación negativa artificial ($\rho_1$ fuertemente negativo) *y* aumenta la varianza. El aumento de varianza es la señal más confiable de las dos, porque un $\rho_1$ negativo moderado es esperable y legítimo tras una diferenciación correcta.
>
> Las varianzas solo son comparables entre series en la **misma escala**: aquí, entre los peldaños 2 a 5, todos en logaritmos.

In [ ]:
ly = np.log(y)

escalera = [
    (y,                         "0) original (escala distinta)", ROJO),
    (ly,                        "1) log", MORADO),
    (ly.diff(S).dropna(),       "2) log + ∇₁₂  (estacional)", VERDE),
    (ly.diff(1).dropna(),       "2b) log + ∇  (solo regular)", GRIS),
    (ly.diff(S).diff(1).dropna(), "3) log + ∇₁₂ + ∇", AZUL),
    (ly.diff(S).diff(1).diff(1).dropna(), "4) log + ∇₁₂ + ∇ + ∇  ← una de más", NARANJA),
]

fig, axes = plt.subplots(len(escalera), 2, figsize=(12.5, 11),
                         gridspec_kw={"width_ratios": [1.7, 1]})
for fila, (serie, titulo, color) in enumerate(escalera):
    axes[fila, 0].plot(np.asarray(serie), color=color, lw=0.9)
    axes[fila, 0].set_title(titulo, loc="left")
    plot_acf(np.asarray(serie), lags=36, ax=axes[fila, 1],
             title="ACF" if fila == 0 else "")
plt.tight_layout(); plt.show()

resumen = pd.DataFrame([diagnostico_estacionariedad(s, t, imprimir=False)
                        for s, t, _ in escalera]).set_index("serie")
print("ESCALERA DE TRANSFORMACIONES")
print(resumen[["n", "p_adf", "p_kpss", "var", "rho1", "veredicto"]]
      .round({"p_adf": 4, "p_kpss": 4, "var": 6, "rho1": 3}).to_string())

In [ ]:
# --- Lectura guiada de la tabla anterior, generada con los valores efectivos ---
v2 = resumen.loc["2) log + ∇₁₂  (estacional)"]
v2b = resumen.loc["2b) log + ∇  (solo regular)"]
v3 = resumen.loc["3) log + ∇₁₂ + ∇"]
v4 = resumen.loc["4) log + ∇₁₂ + ∇ + ∇  ← una de más"]

print(f'''
CÓMO LEER LA ESCALERA
─────────────────────────────────────────────────────────────────────────────
Peldaño 2  (log + ∇₁₂)   ADF p = {v2['p_adf']:.4f} · KPSS p = {v2['p_kpss']:.2f} · var = {v2['var']:.2e}
   Los picos de los rezagos 12, 24 y 36 desaparecen de la ACF: la estacionalidad
   fue removida. Pero ADF NO rechaza al 5% y la ACF decae con lentitud: queda
   deriva en el nivel. La transformación es necesaria y todavía insuficiente.

Peldaño 2b (log + ∇)     ADF p = {v2b['p_adf']:.4f} · var = {v2b['var']:.2e}
   Diferenciar solo de forma regular deja intacta la estacionalidad y produce
   una varianza {v2b['var'] / v2['var']:.1f} veces mayor que el peldaño 2. En una serie
   mensual con estacionalidad marcada, el orden natural es ∇₁₂ primero.

Peldaño 3  (log + ∇₁₂ + ∇)  ADF p = {v3['p_adf']:.4f} · KPSS p = {v3['p_kpss']:.2f} · var = {v3['var']:.2e}
   Ambas pruebas concuerdan en estacionariedad Y la varianza CAE respecto del
   peldaño 2 ({v2['var']:.2e} → {v3['var']:.2e}). El ρ₁ = {v3['rho1']:+.3f} no es alarma:
   es la firma MA(1) característica de una serie correctamente diferenciada.

Peldaño 4  (una diferencia de más)  var = {v4['var']:.2e} · ρ₁ = {v4['rho1']:+.3f}
   La varianza SUBE ({v3['var']:.2e} → {v4['var']:.2e}, un {100 * (v4['var'] / v3['var'] - 1):.0f}% más) y
   ρ₁ se vuelve fuertemente negativo. Es sobre-diferenciación: ADF sigue
   rechazando —una serie sobre-diferenciada también es estacionaria— y por eso
   la prueba por sí sola NO detecta el problema. El criterio de varianza sí.

CONCLUSIÓN: d = 1, D = 1, s = 12, sobre la serie en logaritmos.
''')

# La serie estacionaria de trabajo para todo el resto del notebook
w = ly.diff(S).diff(1).dropna().rename("w")
print(f"Serie estacionaria w_t = ∇∇₁₂ log(X_t):  {len(w)} observaciones "
      f"({len(y)} − {S} − 1), desde {w.index.min():%b-%Y}.")

> ### 📋 Decisión 2 — Órdenes de diferenciación
>
> | Elemento | Contenido |
> |---|---|
> | **Decisión** | $d = 1$, $D = 1$ con $s = 12$, aplicadas sobre $\log X_t$. La serie de trabajo es $w_t = \nabla \nabla_{12} \log X_t$. |
> | **Evidencia** | ADF y KPSS concuerdan en estacionariedad únicamente en el peldaño 3; la varianza alcanza allí su mínimo entre las series comparables; los picos estacionales de la ACF desaparecen tras $\nabla_{12}$. |
> | **Alternativa descartada** | $d=2$: la varianza aumenta un 170% y $\rho_1$ cae a $-0{,}67$, ambas señales de sobre-diferenciación. También se descartó $d=1,\ D=0$: deja intactos los picos en 12, 24 y 36. |
> | **Consecuencia** | Se pierden $s + d = 13$ observaciones. Los modelos de §5 se especifican con estos $d$ y $D$ fijos, lo que además los hace **comparables por AIC** entre sí (§6.1). |

🧭 **Pausa analítica**

1. En el peldaño 2 el KPSS no rechaza la estacionariedad pero el ADF tampoco la confirma. Según la matriz de decisión de §4.5, ¿en qué celda cae ese resultado y qué acción recomienda?
2. ¿Por qué una serie **sobre-diferenciada** sigue pasando la prueba ADF? ¿Qué le dice eso sobre el alcance de las pruebas formales?
3. El logaritmo se aplicó *antes* de diferenciar. Justifique el orden: ¿qué ocurriría si se diferenciara primero y se tomara el logaritmo después?
4. La serie del caso transversal tiene un quiebre exógeno (COVID-19) que dura casi dos años. La diferencia estacional compara cada mes con el mismo mes del año anterior. ¿Qué observaciones de $w_t$ quedarán distorsionadas y cuántas serán?

*Escriba su respuesta aquí (doble clic para editar):*

---

---
# 5. Identificación y estimación de modelos

### 5.1 La familia completa, en una sola notación

Es necesario reconocer y saber estimar cinco especificaciones. Todas son **casos particulares de la misma ecuación**, escrita con el operador de rezago. Sea $w_t$ la serie ya estacionaria y $\varepsilon_t$ ruido blanco de varianza $\sigma^2$:

| Modelo | Ecuación | Parámetros | Firma en ACF / PACF |
|---|---|---|---|
| **AR($p$)** | $\phi_p(B)\,w_t = \varepsilon_t$ | $\phi_1,\dots,\phi_p$ | ACF decae; **PACF se corta** en $p$ |
| **MA($q$)** | $w_t = \theta_q(B)\,\varepsilon_t$ | $\theta_1,\dots,\theta_q$ | **ACF se corta** en $q$; PACF decae |
| **ARMA($p,q$)** | $\phi_p(B)\,w_t = \theta_q(B)\,\varepsilon_t$ | ambos | Ambas decaen |
| **ARIMA($p,d,q$)** | $\phi_p(B)(1-B)^d X_t = \theta_q(B)\,\varepsilon_t$ | + orden $d$ | ARMA sobre $\nabla^d X_t$ |
| **SARIMA($p,d,q$)($P,D,Q$)$_s$** | $\phi_p(B)\,\Phi_P(B^s)\,(1-B)^d (1-B^s)^D X_t = \theta_q(B)\,\Theta_Q(B^s)\,\varepsilon_t$ | + estacionales | Estructura en los rezagos $s, 2s, \dots$ |

donde $\phi_p(B) = 1 - \phi_1 B - \dots - \phi_p B^p$ y $\theta_q(B) = 1 + \theta_1 B + \dots + \theta_q B^q$ (convención de `statsmodels`; otros textos usan signo negativo en $\theta$).

> ### 💡 La idea que unifica todo
> **ARIMA no es un modelo nuevo: es un ARMA aplicado a la serie diferenciada.** La "I" de *integrated* solo registra cuántas diferencias hubo que tomar para llegar allí. Por eso el trabajo del §4 no fue un preámbulo: fue la mitad del modelamiento. Y por eso los órdenes $p$ y $q$ se leen en la ACF y la PACF **de $w_t$**, no de la serie original.
>
> Lo verificaremos numéricamente en §5.5: ajustar un ARMA$(p,q)$ a $w_t$ y ajustar un SARIMA$(p,1,q)(P,1,Q)_{12}$ a $\log X_t$ entregan **los mismos coeficientes**.

### Condiciones de validez

- **Estacionariedad (parte AR):** todas las raíces de $\phi_p(B)=0$ deben estar **fuera** del círculo unitario ($|z| > 1$).
- **Invertibilidad (parte MA):** todas las raíces de $\theta_q(B)=0$ deben estar fuera del círculo unitario. Sin invertibilidad el modelo no puede escribirse como AR($\infty$) y los pronósticos no son únicos.

`statsmodels` las impone por defecto; verificaremos las raíces en §5.6 en lugar de darlas por buenas.

### 5.2 Lectura de la ACF y la PACF de la serie estacionaria

Regla de trabajo para series estacionales: **la ACF y la PACF se leen en dos bloques separados**.

- **Bloque regular:** rezagos $1$ a $s-1$. Determina $p$ y $q$.
- **Bloque estacional:** rezagos $s, 2s, 3s, \dots$ (12, 24, 36). Determina $P$ y $Q$.

Las bandas de confianza al 95% bajo la hipótesis de ruido blanco son $\pm 1{,}96/\sqrt{n}$. Con 36 rezagos, **espere que uno o dos crucen la banda por puro azar**: una barra aislada apenas significativa no es evidencia de estructura.

In [ ]:
n_w = len(w)
banda = 1.96 / np.sqrt(n_w)

fig, axes = plt.subplots(2, 1, figsize=(12.5, 6.4))
plot_acf(w.to_numpy(), lags=36, ax=axes[0], title="")
plot_pacf(w.to_numpy(), lags=36, ax=axes[1], method="ywm", title="")
for ax, nombre in zip(axes, ["ACF", "PACF"]):
    for k in (S, 2 * S, 3 * S):
        ax.axvline(k, color=VERDE, ls=":", lw=1.2)
    ax.set_title(f"{nombre} de w_t = ∇∇₁₂ log(X_t)   ·   líneas verdes: rezagos estacionales 12, 24, 36",
                 loc="left")
    ax.set_xlabel("rezago (meses)")
plt.tight_layout(); plt.show()

r_acf = acf(w.to_numpy(), nlags=36)
r_pacf = pacf(w.to_numpy(), nlags=36, method="ywm")
tabla_corr = pd.DataFrame({"ACF": r_acf[1:15], "PACF": r_pacf[1:15]},
                          index=pd.RangeIndex(1, 15, name="rezago"))
tabla_corr["|ACF|>banda"] = np.abs(tabla_corr["ACF"]) > banda
tabla_corr["|PACF|>banda"] = np.abs(tabla_corr["PACF"]) > banda

print(f"n = {n_w}   banda 95% = ±{banda:.3f}\n")
print(tabla_corr.round(3).to_string())
print(f"\nRezago 12  → ACF {r_acf[12]:+.3f} | PACF {r_pacf[12]:+.3f}")
print(f"Rezago 24  → ACF {r_acf[24]:+.3f} | PACF {r_pacf[24]:+.3f}")

### 5.3 De los correlogramas a los candidatos

**Bloque regular.** La ACF presenta un pico negativo grande en el rezago 1 ($-0{,}34$, el doble de la banda) y luego cruza la banda de forma marginal en los rezagos 3 y 9. La PACF decae con signos alternos. El patrón dominante *ACF se corta en 1 / PACF decae* es la firma de un **MA(1) regular**: $q = 1$, $p = 0$.

Sobre los cruces de los rezagos 3 y 9: con 36 rezagos evaluados al 5%, **se esperan entre uno y dos cruces por puro azar**, y ambos apenas superan la banda. Tratarlos como estructura llevaría a un modelo con términos en rezagos aislados, difícil de justificar teóricamente y propenso al sobreajuste. La lectura prudente es atribuirlos al error muestral —pero no darlo por hecho: el candidato M4 existe precisamente para contrastar esa hipótesis con datos.

**Bloque estacional.** En el rezago 12 la ACF tiene un pico negativo pronunciado, y en 24 y 36 no queda nada significativo. La PACF muestra un valor grande en 12 y decae en los múltiplos siguientes. De nuevo *ACF corta / PACF decae*, ahora en la escala estacional: **MA(1) estacional**, $Q = 1$, $P = 0$.

La lectura conduce directamente a $SARIMA(0,1,1)(0,1,1)_{12}$, el **modelo de aerolínea**. Pero una lectura de correlograma es una **hipótesis, no una conclusión**: los picos son estimaciones con error muestral y la firma "se corta" nunca es tan nítida en datos reales como en los libros. Por eso no estimamos un modelo, sino una **familia de candidatos** que incluye deliberadamente las lecturas alternativas plausibles.

| Candidato | Especificación | Por qué está en la lista |
|---|---|---|
| M1 | $SARIMA(0,1,1)(0,1,1)_{12}$ | Lectura directa de los correlogramas. Modelo de aerolínea. |
| M2 | $SARIMA(1,1,0)(0,1,1)_{12}$ | Lectura alternativa: si el pico de la ACF en 1 se interpreta como decaimiento AR. |
| M3 | $SARIMA(1,1,1)(0,1,1)_{12}$ | Mixto: contrasta si añadir el término AR aporta sobre M1. |
| M4 | $SARIMA(0,1,2)(0,1,1)_{12}$ | Extensión mínima del bloque MA regular: contrasta si $q=1$ basta o si los cruces de los rezagos bajos son estructura. |
| M5 | $SARIMA(0,1,1)(1,1,0)_{12}$ | Lectura estacional alternativa: AR estacional en vez de MA. |
| M6 | $SARIMA(1,1,0)(1,1,0)_{12}$ | Puramente autorregresivo en ambos bloques. Contraste extremo. |
| M7 | $SARIMA(1,1,1)(1,1,1)_{12}$ | Sobre-parametrizado a propósito: sirve para ver si el AIC penaliza. |
| M8 | $ARIMA(0,1,1)$ *(sin componente estacional)* | Contrafactual: cuánto cuesta ignorar la estacionalidad. |

> ### ⚠️ Lo que **no** debe hacer
> Recorrer una grilla exhaustiva de $(p,d,q)(P,D,Q)$ y quedarse con el mínimo AIC. Con suficientes especificaciones alguna gana por azar, y el criterio deja de tener garantía. La grilla acotada y **justificada por la evidencia exploratoria** es lo que tiene valor metodológico; el mínimo global de una búsqueda ciega, no.

### 5.4 Partición cronológica y modelos de referencia

Antes de estimar, dos decisiones que condicionan todo lo que sigue.

**La partición es cronológica, nunca aleatoria.** Un `train_test_split` con `shuffle=True` mezcla el futuro con el pasado y produce métricas optimistas que no significan nada. Reservamos los **últimos 12 meses (1960)** como conjunto de prueba, por dos razones: coincide con el horizonte operacional del caso —el compromiso de capacidad es a doce meses— y deja un ciclo estacional completo para evaluar.

**La referencia trivial define la vara.** Un modelo no se evalúa contra cero, se evalúa contra lo que la organización ya hace. Aquí, contra dos procedimientos sin parámetros:

- **Naive estacional:** $\hat{X}_{t+h} = X_{t+h-12}$. El mismo mes del año anterior.
- **Naive estacional con deriva:** el mismo mes del año anterior multiplicado por el crecimiento interanual medio observado en el entrenamiento. *Es exactamente el procedimiento que la Gerencia usa hoy* (§0).

Si el SARIMA no supera a estas dos referencias, no hay nada que recomendar.

In [ ]:
H = 12                                     # horizonte de prueba = horizonte operacional
tr_log, te_log = ly.iloc[:-H], ly.iloc[-H:]        # escala logaritmica (modelamiento)
tr_org, te_org = y.iloc[:-H], y.iloc[-H:]          # escala original (evaluacion)

print(f"Entrenamiento: {len(tr_log):>3} obs  {tr_log.index.min():%b-%Y} a {tr_log.index.max():%b-%Y}")
print(f"Prueba:        {len(te_log):>3} obs  {te_log.index.min():%b-%Y} a {te_log.index.max():%b-%Y}")
print("\nEl conjunto de prueba queda SELLADO: no se usa para elegir órdenes ni transformaciones.")

# --- Modelos de referencia, evaluados en escala original ---
naive_est = tr_org.to_numpy()[-S:]                                  # mismo mes del año anterior
g_medio = np.mean(tr_org.to_numpy()[S:] / tr_org.to_numpy()[:-S] - 1)
naive_deriva = naive_est * (1 + g_medio)

print(f"\nCrecimiento interanual medio en el entrenamiento: {100 * g_medio:.2f}% anual")


def metricas(real, pred, denom_mase):
    '''RMSE, MAE, MAPE y MASE en la escala original de los datos.'''
    real, pred = np.asarray(real, float), np.asarray(pred, float)
    e = real - pred
    return {"RMSE": np.sqrt(np.mean(e ** 2)),
            "MAE": np.mean(np.abs(e)),
            "MAPE (%)": 100 * np.mean(np.abs(e / real)),
            "MASE": np.mean(np.abs(e)) / denom_mase}


# MASE: escala el error por el del naive estacional DENTRO del entrenamiento.
# MASE < 1 significa "mejor que el naive estacional"; MASE > 1, peor.
denom_mase = np.mean(np.abs(tr_org.to_numpy()[S:] - tr_org.to_numpy()[:-S]))

referencias = pd.DataFrame({
    "Naive estacional": metricas(te_org, naive_est, denom_mase),
    "Naive estacional con deriva": metricas(te_org, naive_deriva, denom_mase),
}).T
print("\nLA VARA A SUPERAR (evaluada sobre 1960, en miles de pasajeros)")
print(referencias.round(3).to_string())

### 5.5 ARIMA es un ARMA sobre la serie diferenciada

Antes de estimar la familia completa, verifiquemos la afirmación de §5.1. Ajustamos a la serie estacionaria $w_t$ tres modelos **sin diferenciación**: un AR(1), un MA(1) y un ARMA(1,1). Luego comprobamos que el ARMA(1,1) sobre $w_t$ reproduce los coeficientes de un $ARIMA(1,1,1)$ estacional equivalente sobre $\log X_t$.

Esto no es un tecnicismo: es la razón por la cual **AR, MA, ARMA, ARIMA y SARIMA no son cinco técnicas distintas** sino una sola con distintos ajustes, y por la cual todo lo que aprendió sobre lectura de correlogramas en la Sesión 1 se aplica sin cambios aquí.

In [ ]:
w_tr = w.loc[:tr_log.index.max()]                  # serie estacionaria, solo entrenamiento

print("Modelos elementales ajustados directamente sobre w_t (sin diferenciar de nuevo):\n")
elementales = {}
for nombre, orden in [("AR(1)", (1, 0, 0)), ("MA(1)", (0, 0, 1)), ("ARMA(1,1)", (1, 0, 1))]:
    fit = SARIMAX(w_tr, order=orden, trend="n").fit(disp=False)
    elementales[nombre] = fit
    coefs = ", ".join(f"{p}={v:+.4f}" for p, v in zip(fit.param_names, np.asarray(fit.params)))
    print(f"  {nombre:<10} AIC = {fit.aic:8.2f}   {coefs}")

# --- Equivalencia: ARMA(1,1) sobre w_t  ==  SARIMA(1,1,1)(0,1,0)_12 sobre log X_t ---
equiv = SARIMAX(tr_log, order=(1, 1, 1), seasonal_order=(0, 1, 0, S)).fit(disp=False)
arma11 = elementales["ARMA(1,1)"]

print("\nCOMPROBACIÓN DE EQUIVALENCIA")
print(f"  ARMA(1,1) sobre w_t                    : ar.L1 = {np.asarray(arma11.params)[0]:+.4f}, "
      f"ma.L1 = {np.asarray(arma11.params)[1]:+.4f}")
print(f"  SARIMA(1,1,1)(0,1,0)_12 sobre log(X_t) : ar.L1 = {np.asarray(equiv.params)[0]:+.4f}, "
      f"ma.L1 = {np.asarray(equiv.params)[1]:+.4f}")
print("\n  Los coeficientes coinciden en las primeras cifras decimales: es el MISMO modelo,")
print("  escrito de dos maneras. La diferencia numérica residual proviene del tratamiento")
print("  de las 13 observaciones iniciales que la diferenciación consume.")
print("\n  Atención: los AIC de estos dos ajustes NO son comparables entre sí, porque")
print("  la verosimilitud se evalúa sobre un número distinto de observaciones. Volvemos")
print("  sobre esto en §6.1.")

### 5.6 Estimación de la familia de candidatos

La estimación es por **máxima verosimilitud** (MLE): se buscan los parámetros que maximizan la probabilidad de haber observado la muestra bajo el modelo, asumiendo errores gaussianos. `statsmodels` implementa el modelo en forma de espacio de estados y evalúa la verosimilitud con el filtro de Kalman, lo que permite tratar de forma exacta las observaciones que la diferenciación deja indeterminadas.

De cada ajuste registramos, además de AIC y BIC:

- si el optimizador **convergió**;
- el **módulo mínimo de las raíces** de los polinomios AR y MA, que debe superar 1;
- cuántos coeficientes resultan **significativos** al 5%.

Un modelo que no converge, o cuyas raíces caen sobre el círculo unitario, se descarta antes de mirar su AIC.

In [ ]:
CANDIDATOS = {
    "M1 SARIMA(0,1,1)(0,1,1)₁₂": ((0, 1, 1), (0, 1, 1, S)),
    "M2 SARIMA(1,1,0)(0,1,1)₁₂": ((1, 1, 0), (0, 1, 1, S)),
    "M3 SARIMA(1,1,1)(0,1,1)₁₂": ((1, 1, 1), (0, 1, 1, S)),
    "M4 SARIMA(0,1,2)(0,1,1)₁₂": ((0, 1, 2), (0, 1, 1, S)),
    "M5 SARIMA(0,1,1)(1,1,0)₁₂": ((0, 1, 1), (1, 1, 0, S)),
    "M6 SARIMA(1,1,0)(1,1,0)₁₂": ((1, 1, 0), (1, 1, 0, S)),
    "M7 SARIMA(1,1,1)(1,1,1)₁₂": ((1, 1, 1), (1, 1, 1, S)),
    "M8 ARIMA(0,1,1) sin estacional": ((0, 1, 1), (0, 0, 0, 0)),
}

ajustes, filas = {}, []
for nombre, (orden, orden_est) in CANDIDATOS.items():
    fit = SARIMAX(tr_log, order=orden, seasonal_order=orden_est,
                  enforce_stationarity=True, enforce_invertibility=True).fit(disp=False)
    ajustes[nombre] = fit

    raices = np.concatenate([np.abs(fit.arroots), np.abs(fit.maroots)]) if (
        len(fit.arroots) + len(fit.maroots)) else np.array([np.inf])
    k = len(fit.params) - 1                              # parametros sin contar sigma^2
    p_vals = np.asarray(fit.pvalues)[:k]

    filas.append({
        "modelo": nombre,
        "k": k,
        "conv": "sí" if fit.mle_retvals.get("converged", True) else "NO",
        "|raíz| mín": raices.min(),
        "coef. signif. (5%)": f"{int((p_vals < 0.05).sum())}/{k}",
        "log-verosim.": fit.llf,
        "AIC": fit.aic,
        "BIC": fit.bic,
        "n efectivo": int(fit.nobs_effective) if hasattr(fit, "nobs_effective") else len(tr_log),
    })

tabla_ajuste = pd.DataFrame(filas).set_index("modelo")
print("ESTIMACIÓN POR MÁXIMA VEROSIMILITUD SOBRE EL ENTRENAMIENTO\n")
print(tabla_ajuste.round({"|raíz| mín": 3, "log-verosim.": 2, "AIC": 2, "BIC": 2}).to_string())
print("\nTodas las raíces superan 1 en módulo: los candidatos son estacionarios e invertibles.")
print("Observe la columna 'n efectivo': M8 no comparte diferenciación con el resto.")

no_conv = list(tabla_ajuste.index[tabla_ajuste["conv"] == "NO"])
if no_conv:
    print(f"\n⚠ El optimizador NO declaró convergencia en: {no_conv}")
    print("  Es el resultado esperable del modelo sobre-parametrizado: con cuatro términos")
    print("  la verosimilitud se vuelve casi plana y el óptimo deja de estar bien definido.")
    print("  Sus coeficientes y su AIC se reportan más abajo solo con fines comparativos;")
    print("  un modelo que no converge NO es candidato a ser seleccionado.")

sub_signif = tabla_ajuste[tabla_ajuste["coef. signif. (5%)"].str.split("/").str[0]
                          != tabla_ajuste["coef. signif. (5%)"].str.split("/").str[1]]
print(f"\nModelos con al menos un coeficiente NO significativo al 5%: {list(sub_signif.index)}")
print("  Un coeficiente no significativo es un parámetro que se paga en el AIC sin aportar")
print("  evidencia de que exista. Es un argumento a favor de la especificación más simple.")

---
# 6. Selección y comparación de modelos

### 6.1 Criterios de información: qué miden y cuándo son comparables

Ambos criterios penalizan la complejidad para evitar que gane siempre el modelo con más parámetros:

$$\mathrm{AIC} = -2\log L + 2k, \qquad \mathrm{BIC} = -2\log L + k\log n, \qquad \mathrm{AICc} = \mathrm{AIC} + \frac{2k(k+1)}{n-k-1}$$

donde $L$ es la verosimilitud maximizada, $k$ el número de parámetros y $n$ el número de observaciones efectivas. **Menor es mejor** en los tres.

| Criterio | Penalización | Tiende a | Cuándo usarlo |
|---|---|---|---|
| **AIC** | $2k$ | Modelos algo más grandes | Objetivo predictivo |
| **BIC** | $k\log n$ (más severa si $n>7$) | Modelos más parsimoniosos | Objetivo explicativo; muestras grandes |
| **AICc** | corrección de muestra finita | Coincide con AIC si $n \gg k$ | Siempre que $n/k < 40$ |

> ### ⚠️ La regla que decide si la comparación es legítima
> **Dos modelos solo son comparables por AIC/BIC si su verosimilitud se calculó sobre exactamente las mismas observaciones y en la misma escala.** Esto implica:
>
> 1. **Mismo $d$ y mismo $D$.** Cada diferencia consume observaciones; con $d$ distintos, $n$ difiere y las verosimilitudes no son homogéneas.
> 2. **Misma transformación.** Comparar el AIC de un modelo en logaritmos con el de uno en la escala original es un error: falta el jacobiano de la transformación. En nuestro caso, $-2\sum\log X_t$.
>
> Por eso M1 a M7 comparten $d=1$, $D=1$ y escala logarítmica: **son comparables entre sí**. M8 tiene $D=0$ y por tanto un $n$ efectivo distinto: **su AIC no entra en la comparación** y solo lo evaluaremos fuera de muestra. Este matiz, que casi ningún informe menciona, es exactamente el tipo de rigor que distingue un análisis sólido de uno correcto pero superficial.

In [ ]:
comparables = tabla_ajuste[tabla_ajuste["n efectivo"] == tabla_ajuste.loc[
    "M1 SARIMA(0,1,1)(0,1,1)₁₂", "n efectivo"]].copy()

n_ef = comparables["n efectivo"]
comparables["AICc"] = comparables["AIC"] + (2 * comparables["k"] * (comparables["k"] + 1)) / (
    n_ef - comparables["k"] - 1)
comparables["ΔAIC"] = comparables["AIC"] - comparables["AIC"].min()
comparables["ΔBIC"] = comparables["BIC"] - comparables["BIC"].min()

print("CRITERIOS DE INFORMACIÓN — solo modelos con igual d, D y escala\n")
print(comparables[["k", "log-verosim.", "AIC", "AICc", "BIC", "ΔAIC", "ΔBIC"]]
      .sort_values("AIC").round(2).to_string())

excluido = tabla_ajuste[tabla_ajuste["n efectivo"] != n_ef.iloc[0]]
print(f"\nExcluido de esta tabla por n efectivo distinto: {list(excluido.index)}")
print(f"  n efectivo del resto: {int(n_ef.iloc[0])} | n efectivo del excluido: {int(excluido['n efectivo'].iloc[0])}")

print('''
CÓMO SE LEE ΔAIC (Burnham y Anderson, 2004)
   Δ < 2      el modelo es indistinguible del mejor: elija por parsimonia.
   2 ≤ Δ < 7  soporte considerablemente menor.
   Δ ≥ 10     el modelo no tiene soporte empírico frente al mejor.
''')

### 6.2 Métricas fuera de muestra y el problema de la retransformación

Los criterios de información miden **ajuste dentro de la muestra**, penalizado. No dicen nada sobre el desempeño futuro. Para eso evaluamos el pronóstico sobre 1960, que el modelo no vio.

Ahora bien, el modelo pronostica $\log X_t$ y la gerencia necesita pasajeros. La retransformación no es inocente:

$$\mathbb{E}[X_{t+h}] = \mathbb{E}\!\left[e^{Z_{t+h}}\right] \;=\; e^{\mathbb{E}[Z_{t+h}]} \cdot e^{\sigma_h^2/2} \;\neq\; e^{\hat{Z}_{t+h}}$$

por la desigualdad de Jensen. Aplicar $\exp$ directamente a la predicción logarítmica entrega la **mediana** de la distribución predictiva, no su media, y por tanto **subestima sistemáticamente** el valor esperado. La corrección multiplicativa $e^{\sigma_h^2/2}$ recupera la media.

> ### 🎯 Cuál de las dos reportar
> Depende de la decisión. Si el costo del error es simétrico, la media es el óptimo. Si es asimétrico —como en el caso: quedarse corto cuesta varias veces más que sobrar— **ni la media ni la mediana son la respuesta**: lo es un cuantil superior de la distribución predictiva. En §8 volvemos sobre esto, que es donde la estadística se convierte en recomendación.
>
> Para los **intervalos** no aplicamos corrección: los límites se retransforman con $\exp$ tal cual, porque los cuantiles sí son invariantes ante transformaciones monótonas.

In [ ]:
def pronosticar(fit, pasos=H, alpha=0.05):
    '''Pronostica en escala logaritmica y retransforma a la escala original.

    Devuelve la media corregida por sesgo, la mediana y los limites del
    intervalo de prediccion, todos en miles de pasajeros.
    '''
    f = fit.get_forecast(steps=pasos)
    media_log = np.asarray(f.predicted_mean)
    var_log = np.asarray(f.var_pred_mean)          # varianza de prediccion en cada horizonte
    ic = np.asarray(f.conf_int(alpha=alpha))
    return {
        "media": np.exp(media_log) * np.exp(var_log / 2),   # E[X] con correccion de sesgo
        "mediana": np.exp(media_log),                        # exp directo
        "inferior": np.exp(ic[:, 0]),
        "superior": np.exp(ic[:, 1]),
    }


filas_oos = []
pronosticos = {}
for nombre, fit in ajustes.items():
    pr = pronosticar(fit)
    pronosticos[nombre] = pr
    m = metricas(te_org, pr["media"], denom_mase)
    m["RMSE sin corregir"] = metricas(te_org, pr["mediana"], denom_mase)["RMSE"]
    m["cobertura IC95% (%)"] = 100 * np.mean(
        (te_org.to_numpy() >= pr["inferior"]) & (te_org.to_numpy() <= pr["superior"]))
    filas_oos.append(pd.Series(m, name=nombre))

tabla_oos = pd.DataFrame(filas_oos)
comparacion = tabla_ajuste[["k", "AIC", "BIC"]].join(tabla_oos)
comparacion.loc["— Naive estacional", ["RMSE", "MAE", "MAPE (%)", "MASE"]] = referencias.loc[
    "Naive estacional", ["RMSE", "MAE", "MAPE (%)", "MASE"]]
comparacion.loc["— Naive estacional con deriva", ["RMSE", "MAE", "MAPE (%)", "MASE"]] = referencias.loc[
    "Naive estacional con deriva", ["RMSE", "MAE", "MAPE (%)", "MASE"]]

print("TABLA COMPARATIVA COMPLETA — ajuste (AIC/BIC) y pronóstico sobre 1960\n")
print(comparacion.round(3).to_string())
print("\nRecuerde: los AIC/BIC de M8 no son comparables con los de M1-M7 (§6.1).")
print("MASE < 1 indica un error menor que el del naive estacional dentro del entrenamiento.")

In [ ]:
# Efecto de la correccion de sesgo, aislado
efecto = tabla_oos["RMSE sin corregir"] - tabla_oos["RMSE"]
print("EFECTO DE LA CORRECCIÓN DE SESGO DE RETRANSFORMACIÓN (RMSE sin corregir − RMSE corregido)")
print(efecto.round(4).to_string())
est = tabla_oos.index[tabla_oos.index != "M8 ARIMA(0,1,1) sin estacional"]
sig2_ref = float(np.asarray(list(ajustes.values())[0].params)[-1])     # sigma^2 de M1
print(f"\nEn los modelos estacionales el efecto es de {efecto[est].abs().mean():.2f} miles de pasajeros "
      f"(σ² ≈ {sig2_ref:.4f} en escala logarítmica: la corrección es de orden "
      f"{100 * (np.exp(sig2_ref / 2) - 1):.2f}% a un paso, y crece con el horizonte).")
print(f"En M8, con σ² mucho mayor, el efecto sube a {abs(efecto['M8 ARIMA(0,1,1) sin estacional']):.1f}.")
print('''
UNA ADVERTENCIA QUE NO DEBE MALINTERPRETARSE
   Aquí la corrección AUMENTA levemente el RMSE de los modelos estacionales. Eso
   no la invalida: la corrección es insesgada por construcción para la MEDIA de
   la distribución predictiva, no minimizadora del error en una muestra concreta
   de doce observaciones. En este año el modelo ya sobreestimaba ligeramente, de
   modo que subir el pronóstico empeoró el ajuste por azar muestral.

   Lo que sí es exigible es DECLARAR cuál de los dos objetos se está reportando
   —media o mediana de la distribución predictiva— y ser consistente. El sesgo
   es sistemático y crece con σ² y con el horizonte: en M8 ya vale 12 mil
   pasajeros, y a 24 o 36 meses deja de ser despreciable en cualquier modelo.
''')

### 6.3 Una partición única no basta

Compare las columnas AIC y RMSE de la tabla anterior: **no ordenan igual los modelos**. Eso no es un error de cálculo, es información. El RMSE sobre 1960 se calcula con **doce residuos de pronóstico**, de modo que su error de estimación es enorme: basta un año atípico para invertir el orden.

Una forma barata de comprobar cuán frágil es ese ranking consiste en repetir la evaluación **moviendo el origen del pronóstico**: entrenamos con los datos hasta un mes $t_0$, pronosticamos doce meses, medimos, avanzamos $t_0$ y repetimos. Es la validación de **origen móvil** (*rolling origin* o *time series cross-validation*), el análogo temporal de la validación cruzada, y cada modelo termina evaluado sobre decenas de pronósticos en lugar de uno.

In [ ]:
def validacion_origen_movil(orden, orden_est, serie_log, h=12, primer_origen=96, paso=3):
    '''Evalua un modelo repitiendo el pronostico desde origenes sucesivos.

    Ventana expansiva: cada origen reestima el modelo con toda la historia
    disponible hasta ese punto. Devuelve el RMSE medio en escala original.
    '''
    errores = []
    for t0 in range(primer_origen, len(serie_log) - h + 1, paso):
        fit = SARIMAX(serie_log.iloc[:t0], order=orden,
                      seasonal_order=orden_est).fit(disp=False)
        f = fit.get_forecast(steps=h)
        pred = np.exp(np.asarray(f.predicted_mean)) * np.exp(np.asarray(f.var_pred_mean) / 2)
        real = np.exp(serie_log.iloc[t0:t0 + h].to_numpy())
        errores.append(np.sqrt(np.mean((real - pred) ** 2)))
    return np.array(errores)


print("Validación de origen móvil (ventana expansiva, h = 12, un origen cada 3 meses)")
print("Puede tardar algo más de un minuto: se reestima cada modelo en cada origen.\n")

filas_rom = []
for nombre, (orden, orden_est) in CANDIDATOS.items():
    e = validacion_origen_movil(orden, orden_est, ly)
    filas_rom.append({"modelo": nombre, "RMSE medio": e.mean(), "desv. entre orígenes": e.std(),
                      "RMSE peor origen": e.max(), "n orígenes": len(e)})

tabla_rom = pd.DataFrame(filas_rom).set_index("modelo").sort_values("RMSE medio")
print(tabla_rom.round(3).to_string())

# Las referencias triviales, evaluadas con el mismo protocolo
err_ne, err_nd = [], []
for t0 in range(96, len(ly) - H + 1, 3):
    hist = np.exp(ly.iloc[:t0].to_numpy())
    real = np.exp(ly.iloc[t0:t0 + H].to_numpy())
    ne = hist[-S:]
    gg = np.mean(hist[S:] / hist[:-S] - 1)
    err_ne.append(np.sqrt(np.mean((real - ne) ** 2)))
    err_nd.append(np.sqrt(np.mean((real - ne * (1 + gg)) ** 2)))
print(f"\n— Naive estacional            RMSE medio = {np.mean(err_ne):7.3f}")
print(f"— Naive estacional con deriva RMSE medio = {np.mean(err_nd):7.3f}")

In [ ]:
fig, ax = plt.subplots(figsize=(11, 3.8))
orden_barras = tabla_rom.sort_values("RMSE medio")
colores = [ROJO if i == 0 else GRIS for i in range(len(orden_barras))]
ax.barh(range(len(orden_barras)), orden_barras["RMSE medio"], color=colores, alpha=0.85)
ax.set_yticks(range(len(orden_barras)))
ax.set_yticklabels(orden_barras.index, fontsize=8)
ax.invert_yaxis()
ax.axvline(np.mean(err_nd), color=AZUL, ls="--", lw=1.6,
           label=f"naive con deriva ({np.mean(err_nd):.1f})")
ax.axvline(np.mean(err_ne), color=VERDE, ls=":", lw=1.6,
           label=f"naive estacional ({np.mean(err_ne):.1f})")
ax.set_xlabel("RMSE medio en validación de origen móvil (miles de pasajeros)")
ax.set_title("Desempeño robusto: 13 orígenes de pronóstico, no uno solo", loc="left")
ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

### 6.4 Decisión: qué modelo y por qué

Reunamos la evidencia en lugar de quedarnos con el mínimo de una sola columna.

1. **Ajuste penalizado.** M1, el modelo de aerolínea, obtiene el mejor AIC y el mejor BIC entre los seis modelos comparables y convergentes, con solo dos parámetros, ambos significativos al 1%. M2 queda a $\Delta\mathrm{AIC} = 0{,}4$ —indistinguible— y M3 y M4 a menos de 2 puntos, pero pagando un parámetro adicional que **no resulta significativo**. M7, deliberadamente sobre-parametrizado, ni siquiera converge y queda 9 puntos de BIC atrás: la penalización está funcionando.
2. **Pronóstico robusto.** En la validación de origen móvil los siete candidatos estacionales quedan dentro de un 6% de RMSE entre sí, y los tres primeros —M3, M4 y M1— dentro de un 1,2%: **son indistinguibles**, sobre todo si se compara esa diferencia con la desviación entre orígenes, que supera las 9 unidades. Todos los candidatos estacionales superan a las dos referencias triviales; M8, que ignora la estacionalidad, queda muy por detrás incluso del naive estacional.
3. **La partición única habría llevado a otra conclusión.** Sobre 1960, el *naive* estacional con deriva —el procedimiento que la Gerencia usa hoy— obtiene un RMSE **menor** que el de M1. Si la evaluación hubiese terminado ahí, la recomendación profesional habría sido no cambiar de procedimiento. La validación de origen móvil, con trece ventanas en lugar de una, invierte el resultado y muestra que 1960 fue un año particularmente favorable para la regla trivial. Esta es la razón por la cual **una partición única no basta para tomar la decisión**, y no un tecnicismo metodológico.
4. **Parsimonia.** Ante desempeños equivalentes, se elige el modelo con menos parámetros. Es el principio de parsimonia de Box y Jenkins, y tiene una justificación práctica: menos parámetros estimados significa menos varianza de estimación y pronósticos más estables al reestimar.
5. **Coherencia con la identificación.** M1 es exactamente el modelo que la lectura de la ACF y la PACF sugirió en §5.3. Que el criterio cuantitativo confirme la lectura cualitativa es la mejor señal de que la identificación fue correcta.

> ### 📋 Decisión 3 — Modelo seleccionado
>
> | Elemento | Contenido |
> |---|---|
> | **Decisión** | $SARIMA(0,1,1)(0,1,1)_{12}$ sobre $\log X_t$ (modelo de aerolínea). |
> | **Evidencia** | Mejor AIC y mejor BIC entre los modelos comparables y convergentes; ambos coeficientes significativos al 1%; RMSE de origen móvil a 1,2% del mejor candidato, diferencia despreciable frente a la desviación entre orígenes; supera a ambas referencias triviales; coincide con la lectura de los correlogramas de §5.3. |
> | **Alternativa descartada** | M3, $SARIMA(1,1,1)(0,1,1)_{12}$: RMSE marginalmente menor en origen móvil, pero un parámetro adicional **no significativo** y peor AIC y BIC. También M2, indistinguible por AIC ($\Delta = 0{,}4$): se prefiere M1 por coherencia con la identificación, que apuntaba a un MA regular y no a un AR. |
> | **Consecuencia** | El pronóstico y todo el diagnóstico de §7 se realizan sobre este modelo. Si el diagnóstico de residuos fallara, se vuelve al paso 1 del ciclo con M3 o M4 como siguientes candidatos. |

🧭 **Pausa analítica**

1. Si el BIC hubiese seleccionado un modelo distinto al AIC, ¿cuál habría preferido en este caso, dado el objetivo del §0? Justifique con la fórmula de penalización.
2. M8 ignora la estacionalidad. Mire su RMSE fuera de muestra frente al de M1: ¿cuánto cuesta, en pasajeros, omitir el componente estacional? ¿Por qué su AIC —que es el más bajo de toda la tabla— no sirve para llegar a esa conclusión?
3. La validación de origen móvil reestima el modelo en cada origen. ¿Qué cambiaría —y por qué sería una versión más optimista— si se estimaran los coeficientes una sola vez con toda la serie y solo se movieran los orígenes?
4. El *naive* con deriva ganó en la partición única y perdió en el origen móvil. Si usted solo hubiese ejecutado la partición única, ¿qué habría recomendado? ¿Qué le dice esto sobre los informes que reportan una sola métrica sobre un solo conjunto de prueba?

In [ ]:
MEJOR = "M1 SARIMA(0,1,1)(0,1,1)₁₂"
modelo = ajustes[MEJOR]
print(modelo.summary())

In [ ]:
pr = pronosticos[MEJOR]
fig, ax = plt.subplots(figsize=(12.5, 4.4))

ax.plot(tr_org.index, tr_org.to_numpy(), color="#333333", lw=1.2, label="entrenamiento (1949-1959)")
ax.plot(te_org.index, te_org.to_numpy(), color=AZUL, lw=2.2, marker="o", ms=3.5,
        label="valores reales de 1960 (sellados)")
ax.plot(te_org.index, pr["media"], color=ROJO, lw=2.2, ls="--", marker="s", ms=3.5,
        label=f"pronóstico {MEJOR.split()[0]}")
ax.fill_between(te_org.index, pr["inferior"], pr["superior"], color=ROJO, alpha=0.14,
                label="intervalo de predicción 95%")
ax.plot(te_org.index, naive_deriva, color=VERDE, lw=1.4, ls=":", label="naive con deriva (procedimiento actual)")

ax.set_xlim(pd.Timestamp("1956-01-01"), te_org.index.max() + pd.Timedelta(days=40))
ax.set_ylabel("miles de pasajeros")
ax.set_title("Evaluación fuera de muestra: 1960 nunca fue visto por el modelo", loc="left")
ax.legend(fontsize=8, loc="upper left")
plt.tight_layout(); plt.show()

detalle = pd.DataFrame({
    "real": te_org.to_numpy(),
    "pronóstico": pr["media"],
    "IC inf": pr["inferior"],
    "IC sup": pr["superior"],
}, index=te_org.index.strftime("%b-%Y"))
detalle["error"] = detalle["real"] - detalle["pronóstico"]
detalle["error %"] = 100 * detalle["error"] / detalle["real"]
detalle["dentro del IC"] = (detalle["real"] >= detalle["IC inf"]) & (detalle["real"] <= detalle["IC sup"])
print(detalle.round(2).to_string())
print(f"\nCobertura empírica del intervalo 95%: {100 * detalle['dentro del IC'].mean():.1f}% "
      f"({int(detalle['dentro del IC'].sum())} de {len(detalle)} meses).")

---
# 7. Diagnóstico de residuos y validación de supuestos

### 7.1 Qué se está probando exactamente

El modelo supone que, una vez extraída toda la estructura lineal, lo que queda es **ruido blanco**: $\varepsilon_t \sim \mathrm{RB}(0, \sigma^2)$, y para que la inferencia y los intervalos sean exactos, además gaussiano. El diagnóstico verifica cuatro afirmaciones, cada una con su consecuencia práctica si falla:

| # | Supuesto | Prueba formal | $H_0$ | Si falla… |
|---|---|---|---|---|
| 1 | **Independencia** (sin autocorrelación) | Ljung–Box | los residuos no están autocorrelacionados | Queda estructura sin modelar. **Es el único bloqueante**: obliga a respecificar. |
| 2 | **Media cero** | $t$ sobre la media | $\mathbb{E}[\varepsilon_t]=0$ | El pronóstico está sesgado; se corrige con una constante. |
| 3 | **Homocedasticidad** | Goldfeld–Quandt, ARCH-LM | varianza constante | El pronóstico puntual sigue siendo válido; **los intervalos no**. |
| 4 | **Normalidad** | Shapiro–Wilk, Jarque–Bera | los residuos son normales | Los intervalos son aproximados; con $n$ grande el pronóstico puntual resiste. |

> ### ⚠️ Dos detalles técnicos que casi todos los informes omiten
>
> **(a) Grados de libertad de Ljung–Box.** El estadístico $Q(K) = n(n+2)\sum_{k=1}^{K}\frac{\hat\rho_k^2}{n-k}$ se distribuye $\chi^2$ con $K - m$ grados de libertad, donde $m$ es el **número de parámetros ARMA estimados**, no con $K$. Usar $K$ hace la prueba demasiado conservadora: se deja pasar autocorrelación real. En `statsmodels` se corrige con el argumento `model_df`.
>
> **(b) Las primeras observaciones no son residuos.** El filtro de Kalman inicializa el estado con una difusa aproximada; los primeros $d + D\cdot s = 1 + 12 = 13$ valores de `resid` son artefactos de arranque, con magnitudes que pueden ser cien veces mayores que el resto. Incluirlos destruye cualquier prueba de normalidad y de varianza. **Se descartan.**
>
> Ambos detalles cambian materialmente las conclusiones. Verifíquelos en la celda siguiente antes de aceptar cualquier diagnóstico.

In [ ]:
m_arma = len(modelo.params) - 1              # parametros ARMA, sin contar sigma^2
n_descartar = 1 + 1 * S                      # d + D*s observaciones de arranque

res_todos = pd.Series(np.asarray(modelo.resid), index=tr_log.index, name="residuos")
res = res_todos.iloc[n_descartar:]

print("POR QUÉ SE DESCARTAN LAS PRIMERAS OBSERVACIONES")
print(f"Primeros {n_descartar + 3} residuos:")
print(np.round(res_todos.to_numpy()[:n_descartar + 3], 4))
print(f"\nDesviación estándar incluyendo el arranque : {res_todos.std():.4f}")
print(f"Desviación estándar descartando {n_descartar} valores : {res.std():.4f}")
print(f"El arranque infla la desviación estándar por un factor de {res_todos.std() / res.std():.1f}.")
print(f"\nResiduos utilizados en el diagnóstico: {len(res)} "
      f"({res.index.min():%b-%Y} a {res.index.max():%b-%Y})")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12.5, 7.2))

# 1. Residuos en el tiempo
axes[0, 0].plot(res.index, res.to_numpy(), color=MORADO, lw=1.0)
axes[0, 0].axhline(0, color="k", lw=0.9)
axes[0, 0].axhline(+2 * res.std(), color=ROJO, ls=":", lw=1.0)
axes[0, 0].axhline(-2 * res.std(), color=ROJO, ls=":", lw=1.0)
axes[0, 0].set_title("Residuos en el tiempo (bandas ±2σ)", loc="left")

# 2. Histograma con densidad normal teorica superpuesta
axes[0, 1].hist(res.to_numpy(), bins=18, density=True, color="#4db6ac", alpha=0.75,
                edgecolor="white")
xx = np.linspace(res.min(), res.max(), 200)
axes[0, 1].plot(xx, stats.norm.pdf(xx, res.mean(), res.std()), color=ROJO, lw=1.8,
                label="normal teórica")
axes[0, 1].set_title("Histograma vs. densidad normal", loc="left")
axes[0, 1].legend(fontsize=8); axes[0, 1].grid(False)

# 3. Grafico Q-Q
sm.qqplot(res.to_numpy(), line="s", ax=axes[1, 0], markerfacecolor=AZUL,
          markeredgecolor=AZUL, alpha=0.6)
axes[1, 0].set_title("Gráfico Q-Q normal", loc="left")

# 4. ACF de los residuos
plot_acf(res.to_numpy(), lags=36, ax=axes[1, 1], title="")
for k in (S, 2 * S, 3 * S):
    axes[1, 1].axvline(k, color=VERDE, ls=":", lw=1.0)
axes[1, 1].set_title("ACF de los residuos (verde: rezagos estacionales)", loc="left")

plt.tight_layout(); plt.show()

In [ ]:
print("=" * 78)
print("BATERÍA FORMAL DE DIAGNÓSTICO — " + MEJOR)
print("=" * 78)

# --- 1. INDEPENDENCIA: Ljung-Box con grados de libertad corregidos ---
print(f"\n1. INDEPENDENCIA — Ljung-Box (H0: sin autocorrelación; gl = K − {m_arma})")
lb = acorr_ljungbox(res, lags=[6, 12, 18, 24, 36], model_df=m_arma, return_df=True)
lb.index.name = "K"
print(lb.round(4).to_string())
lb_ok = bool((lb["lb_pvalue"].dropna() > 0.05).all())
print(f"   → {'No se rechaza H0 en ningún K: residuos compatibles con incorrelación.' if lb_ok else 'SE RECHAZA: queda estructura sin modelar.'}")

# Comparacion con la version SIN corregir los grados de libertad
lb_mal = acorr_ljungbox(res, lags=[12], model_df=0, return_df=True)
print(f"   Nota: con gl = K (sin corregir) el p-valor en K=12 sería {lb_mal['lb_pvalue'].iloc[0]:.4f} "
      f"en vez de {lb['lb_pvalue'].loc[12]:.4f}: la prueba mal especificada es más permisiva.")

# --- 2. MEDIA CERO ---
t_stat, p_media = stats.ttest_1samp(res.to_numpy(), 0.0)
print(f"\n2. MEDIA CERO — prueba t (H0: media = 0)")
print(f"   media = {res.mean():+.5f}  ·  t = {t_stat:+.3f}  ·  p = {p_media:.4f}")
print(f"   → {'No se rechaza H0: el pronóstico no presenta sesgo sistemático.' if p_media > 0.05 else 'SE RECHAZA: hay sesgo; considere incluir una constante.'}")

# --- 3. HOMOCEDASTICIDAD ---
print(f"\n3. HOMOCEDASTICIDAD")
arch_lm, arch_p = het_arch(res.to_numpy(), nlags=12)[:2]
print(f"   ARCH-LM(12)  (H0: sin heterocedasticidad condicional)  LM = {arch_lm:.3f}  p = {arch_p:.4f}")
gq_f, gq_p = het_goldfeldquandt(res.to_numpy(), np.ones((len(res), 1)),
                                split=0.5, drop=0.1, alternative="two-sided")[:2]
print(f"   Goldfeld-Quandt (H0: varianza igual en ambas mitades)   F = {gq_f:.3f}  p = {gq_p:.4f}")
mitad = len(res) // 2
print(f"   desv. est. primera mitad = {res.to_numpy()[:mitad].std():.4f}  ·  "
      f"segunda mitad = {res.to_numpy()[mitad:].std():.4f}")
print("   desviación estándar de los residuos por año:")
print("   " + res.groupby(res.index.year).std().round(4).to_string().replace("\n", "\n   "))

# --- 4. NORMALIDAD ---
print(f"\n4. NORMALIDAD")
sw_w, sw_p = stats.shapiro(res.to_numpy())
jb_s, jb_p = stats.jarque_bera(res.to_numpy())[:2]
print(f"   Shapiro-Wilk   W = {sw_w:.4f}  p = {sw_p:.4f}")
print(f"   Jarque-Bera    JB = {jb_s:.4f}  p = {jb_p:.4f}")
print(f"   asimetría = {stats.skew(res.to_numpy()):+.3f}   curtosis (exceso) = {stats.kurtosis(res.to_numpy()):+.3f}")
print(f"   → {'No se rechaza la normalidad.' if min(sw_p, jb_p) > 0.05 else 'SE RECHAZA la normalidad.'}")

print("\n" + "=" * 78)
veredicto = {"Independencia": lb_ok, "Media cero": p_media > 0.05,
             "Homocedasticidad": min(arch_p, gq_p) > 0.05,
             "Normalidad": min(sw_p, jb_p) > 0.05}
for k, v in veredicto.items():
    print(f"   {k:<20} {'✔ se cumple' if v else '✘ NO se cumple'}")
print("=" * 78)

### 7.2 Interpretación: tres supuestos se cumplen y uno no

Este es el resultado que hace útil el ejercicio, porque **no es limpio**, y lo que corresponde es concluir sobre la validez del modelo, no declarar que todo salió bien.

**Independencia — se cumple.** Ljung–Box no rechaza en ninguno de los cinco valores de $K$ evaluados, con grados de libertad corregidos, y la ACF de los residuos no muestra picos significativos ni en el bloque regular ni en los rezagos 12, 24 y 36. **No queda estructura lineal sin modelar.** Este era el supuesto bloqueante y está satisfecho: el ciclo de Box–Jenkins puede cerrarse.

**Media cero — se cumple.** El pronóstico no arrastra sesgo sistemático.

**Normalidad — se cumple.** Shapiro–Wilk y Jarque–Bera coinciden en no rechazar; el gráfico Q-Q se pega a la recta salvo en los extremos. Los intervalos de predicción tienen respaldo distribucional.

**Homocedasticidad — no se cumple.** El propio `summary()` del modelo lo anticipa en la línea `Prob(H)`, que casi nadie lee. Conviene leer las dos pruebas por separado, porque miden cosas distintas:

- **ARCH-LM no rechaza**: no hay *agrupamiento de volatilidad*, es decir, la varianza no depende de los errores recientes. No estamos ante un fenómeno de tipo GARCH.
- **Goldfeld–Quandt sí rechaza**: la varianza de la primera mitad de la muestra es sustancialmente mayor que la de la segunda. La tabla por año lo confirma: la dispersión de los residuos **decrece de forma monótona** a lo largo del periodo.

**Diagnóstico de la causa.** La transformación logarítmica corresponde a $\lambda = 0$, y en §4.4 estimamos $\hat{\lambda} \approx 0{,}15$. El logaritmo es, por tanto, **ligeramente más fuerte de lo que la serie necesita**: sobre-corrige la varianza y, como el nivel crece con el tiempo, deja residuos relativamente más pequeños al final de la muestra. La pequeña diferencia que anotamos en la Decisión 1 reaparece aquí, exactamente donde el diagnóstico debía encontrarla.

**Consecuencias, en orden de importancia para la decisión del caso:**

1. El **pronóstico puntual sigue siendo válido**: la heterocedasticidad no sesga la media condicional.
2. Los **intervalos son conservadores en el tramo reciente**: el modelo estima una $\sigma^2$ promedio de todo el periodo, mayor que la varianza efectiva de los últimos años, de modo que el rango reportado es más ancho de lo que la dinámica reciente justifica. Note que la **cobertura empírica no sirve para detectar esto**: con doce meses de prueba, cada mes vale 8,3 puntos porcentuales y la estimación es demasiado ruidosa para distinguir un 92% de un 98%. La evidencia está en la tabla de desviaciones por año, no en el conteo de aciertos.
3. Para el caso, un intervalo **demasiado ancho** es un error del lado seguro: lleva a comprometer capacidad de más, que es el error barato. Pero debe **declararse**, no ocultarse, porque una gerencia que descubre que el rango era artificialmente amplio deja de creer en los siguientes.

> ### 🛠️ Qué hacer cuando un supuesto falla
>
> | Supuesto que falla | Diagnóstico | Acción |
> |---|---|---|
> | Autocorrelación residual en rezagos bajos | Falta estructura regular | Aumentar $p$ o $q$; volver al paso 1 |
> | Autocorrelación en el rezago $s$ | Falta estructura estacional | Revisar $P$, $Q$ o $D$ |
> | Heterocedasticidad condicional (ARCH-LM rechaza) | Agrupamiento de volatilidad | Modelo GARCH sobre los residuos (**Unidad 4**) |
> | Heterocedasticidad monótona (GQ rechaza, ARCH no) | Transformación mal calibrada | Reestimar $\lambda$ de Box–Cox; declarar el efecto sobre los intervalos |
> | No normalidad con colas pesadas | Atípicos o quiebres | Identificarlos; variables indicadoras; intervalos por *bootstrap* |
>
> En este caso la falla es de la penúltima fila y **no es bloqueante**: se documenta como limitación y se cuantifica su efecto sobre los intervalos. Reespecificar con Box–Cox agregaría un parámetro estimado y complicaría la retransformación a cambio de una mejora marginal. Es una decisión defendible en ambos sentidos, y lo importante es que esté **argumentada**.

🧭 **Pausa analítica**

1. Ljung–Box con $K=12$ y $m=2$ tiene 10 grados de libertad. ¿Por qué usar 12 haría la prueba *más permisiva* y no más exigente?
2. ARCH-LM no rechaza y Goldfeld–Quandt sí. Explique con sus palabras qué mide cada una y por qué no es contradictorio.
3. Los intervalos resultaron conservadores. Si el costo de quedarse corto fuese, en cambio, **menor** que el de sobrar, ¿seguiría siendo un error del lado seguro?
4. Suponga que Ljung–Box hubiese rechazado con un pico en el rezago 12 de la ACF de los residuos. ¿Qué órdenes modificaría y por qué?

*Escriba su respuesta aquí:*

---

---
# 8. Del modelo a la recomendación

El ciclo está cerrado: la serie fue estacionarizada con justificación, los órdenes se identificaron a partir de la evidencia, el modelo fue seleccionado entre candidatos y sus residuos superaron el diagnóstico. Queda usarlo.

Dos ajustes antes de pronosticar el futuro real:

1. **Reestimar con toda la serie.** La partición existió para *evaluar*. Una vez elegido el modelo, retener los últimos doce meses sería desperdiciar la información más reciente, que es la más relevante para el pronóstico. Se conservan los órdenes; se reestiman los coeficientes.
2. **Traducir el pronóstico a la decisión.** El caso tiene costos asimétricos: quedarse corto cuesta varias veces más que sobrar. El volumen a comprometer no es la media de la distribución predictiva, sino un **cuantil superior**. Si el costo de subestimar es $c_u$ y el de sobreestimar es $c_o$, el cuantil óptimo es $q^\* = c_u/(c_u+c_o)$ —el resultado clásico del *problema del vendedor de diarios*—. Con $c_u = 3c_o$, $q^\* = 0{,}75$.

In [ ]:
# --- Reestimacion con la serie completa ---
modelo_final = SARIMAX(ly, order=CANDIDATOS[MEJOR][0],
                       seasonal_order=CANDIDATOS[MEJOR][1]).fit(disp=False)

print("Coeficientes: entrenamiento vs. serie completa\n")
comp_coef = pd.DataFrame({
    "entrenamiento (132 obs)": np.asarray(modelo.params),
    "serie completa (144 obs)": np.asarray(modelo_final.params),
}, index=modelo.param_names)
print(comp_coef.round(4).to_string())
print("\nLos coeficientes son estables al agregar un año de datos: señal de que el modelo")
print("no está sobreajustado a un tramo particular de la muestra.")

# --- Pronostico de 1961 ---
HORIZONTE = 12
f = modelo_final.get_forecast(steps=HORIZONTE)
media_log, var_log = np.asarray(f.predicted_mean), np.asarray(f.var_pred_mean)
sd_log = np.sqrt(var_log)

idx_fut = pd.date_range(y.index.max() + pd.offsets.MonthBegin(1), periods=HORIZONTE, freq="MS")
C_SUB, C_SOB = 3.0, 1.0                      # costo de quedarse corto vs. de sobrar
q_opt = C_SUB / (C_SUB + C_SOB)

plan = pd.DataFrame({
    "pronóstico (media)": np.exp(media_log) * np.exp(var_log / 2),
    "IC 95% inferior": np.exp(media_log - 1.96 * sd_log),
    "IC 95% superior": np.exp(media_log + 1.96 * sd_log),
    f"cuantil {q_opt:.0%} (a comprometer)": np.exp(media_log + stats.norm.ppf(q_opt) * sd_log),
}, index=idx_fut.strftime("%b-%Y"))

print(f"\nPRONÓSTICO 1961 — miles de pasajeros   (c_sub = {C_SUB:.0f}·c_sob ⇒ cuantil {q_opt:.0%})\n")
print(plan.round(1).to_string())
print(f"\nTotal anual esperado          : {plan['pronóstico (media)'].sum():,.0f} mil pasajeros")
print(f"Total anual a comprometer     : {plan.iloc[:, 3].sum():,.0f} mil pasajeros "
      f"({100 * (plan.iloc[:, 3].sum() / plan['pronóstico (media)'].sum() - 1):.1f}% sobre la media)")
print(f"Amplitud media del IC 95%     : ±{(100 * (plan['IC 95% superior'] - plan['IC 95% inferior']) / (2 * plan['pronóstico (media)'])).mean():.1f}% del valor pronosticado")

In [ ]:
fig, ax = plt.subplots(figsize=(12.5, 4.4))
ax.plot(y.index, y.to_numpy(), color="#333333", lw=1.2, label="histórico 1949-1960")
ax.plot(idx_fut, plan["pronóstico (media)"], color=ROJO, lw=2.2, marker="o", ms=3.5,
        label="pronóstico 1961")
ax.fill_between(idx_fut, plan["IC 95% inferior"], plan["IC 95% superior"],
                color=ROJO, alpha=0.14, label="intervalo de predicción 95%")
ax.plot(idx_fut, plan.iloc[:, 3], color=AZUL, lw=1.6, ls="--",
        label=f"volumen a comprometer (cuantil {q_opt:.0%})")
ax.axvline(y.index.max(), color=GRIS, ls=":", lw=1.2)
ax.set_xlim(pd.Timestamp("1954-01-01"), idx_fut.max() + pd.Timedelta(days=40))
ax.set_ylabel("miles de pasajeros")
ax.set_title("Pronóstico operacional con el modelo validado", loc="left")
ax.legend(fontsize=8, loc="upper left")
plt.tight_layout(); plt.show()

print('''
LO QUE SE LE INFORMA A LA GERENCIA
─────────────────────────────────────────────────────────────────────────────
1. El modelo supera al procedimiento actual (mismo mes del año anterior más el
   crecimiento medio) en la validación de origen móvil, evaluada sobre trece
   ventanas de pronóstico independientes y no sobre un único año favorable.
2. El intervalo entregado es conservador por una limitación identificada y
   declarada: la varianza de los residuos decrece a lo largo de la muestra
   (§7.2). El rango real es algo más estrecho que el reportado.
3. El volumen recomendado NO es el pronóstico puntual. Bajo el supuesto de que
   quedarse corto cuesta el triple que sobrar, corresponde comprometer el
   cuantil 75% de la distribución predictiva. Cambiar ese supuesto de costos
   cambia la recomendación, y esa relación debe validarla el área comercial.
4. El pronóstico supone que no ocurren quiebres estructurales. Cualquier cambio
   de red, de flota o del entorno regulatorio invalida la proyección y obliga a
   reestimar.
''')

---
# 9. Conclusiones

El objetivo de la sesión no era pronosticar pasajeros: era **construir una cadena de decisiones auditable** en la que cada eslabón se apoya en evidencia y ninguno se justifica por costumbre.

| # | Decisión | Evidencia que la sostiene | Sección |
|---|---|---|---|
| 1 | Logaritmo como transformación estabilizadora | Coeficiente de variación anual estable; $\hat{\lambda}_{\text{BC}} \approx 0{,}15$ | §4.4 |
| 2 | $d = 1$, $D = 1$, $s = 12$ | Concordancia ADF-KPSS y mínimo de varianza en ese peldaño | §4.6 |
| 3 | Candidatos con $q=1$, $Q=1$ y sus lecturas alternativas | ACF se corta en 1 y en 12; PACF decae en ambos bloques | §5.3 |
| 4 | $SARIMA(0,1,1)(0,1,1)_{12}$ | Mejor AIC y BIC; origen móvil dentro del 1% del mejor; parsimonia | §6.4 |
| 5 | Modelo validado y utilizable | Ljung–Box, media cero y normalidad no se rechazan | §7.1 |
| 6 | Limitación declarada: heterocedasticidad monótona | Goldfeld–Quandt rechaza; ARCH-LM no | §7.2 |
| 7 | Comprometer el cuantil 75%, no la media | Costos asimétricos del caso ($c_u = 3c_o$) | §8 |

### Lo que se llevó de esta sesión

1. **ARIMA es ARMA sobre la serie diferenciada.** Las cinco especificaciones estudiadas son una sola ecuación con distintos términos activados.
2. **La identificación produce hipótesis, no conclusiones.** Se estima una familia de candidatos y se deja que la evidencia decida.
3. **AIC y RMSE responden preguntas distintas** y no tienen por qué coincidir. Ninguno de los dos, por sí solo, selecciona un modelo.
4. **Un ranking basado en una partición única es frágil.** El origen móvil cuesta un minuto de cómputo y cambia la calidad de la conclusión.
5. **El diagnóstico de residuos es la única prueba de que el modelo es utilizable**, y un supuesto que falla no siempre invalida el modelo: obliga a declarar una limitación y a acotar su efecto.
6. **Un modelo se justifica superando a la alternativa trivial.** Si el naive con deriva hubiese ganado, la recomendación profesional habría sido usar el naive con deriva.

---
# 10. Errores frecuentes en la unidad

| Error | Por qué ocurre | Qué hacer |
|---|---|---|
| Leer la ACF/PACF de la serie **original** para elegir $p$ y $q$ | Se salta el paso 1 del ciclo | Identificar siempre sobre la serie ya estacionaria |
| Usar `np.diff(x, n=12)` como diferencia estacional | El nombre del argumento induce al error | `x.diff(12)`; verificar la longitud del resultado |
| Diferenciar "por si acaso" hasta que ADF rechace | ADF también acepta series sobre-diferenciadas | Criterio de varianza: deténgase cuando la varianza suba |
| Comparar AIC entre modelos con distinto $d$, $D$ o transformación | La librería imprime el número sin advertencia | Comparar solo con igual diferenciación y escala; si no, usar métricas fuera de muestra |
| Retransformar con `np.exp()` sin corregir el sesgo | Parece la operación inversa evidente | Multiplicar por $e^{\sigma_h^2/2}$ para obtener la media; declarar cuál se reporta |
| Ljung–Box con `model_df=0` | Es el valor por defecto | Descontar los parámetros ARMA estimados |
| Diagnosticar con los residuos completos | `resid` incluye el arranque del filtro | Descartar los primeros $d + D\cdot s$ valores |
| Buscar el mínimo AIC en una grilla exhaustiva | Automatización sin criterio | Grilla acotada y justificada por la evidencia exploratoria |
| Evaluar con `train_test_split(shuffle=True)` | Hábito del aprendizaje supervisado | La partición debe ser **cronológica** |
| Declarar el modelo válido porque el $R^2$ es alto | Analogía con regresión | En series con tendencia el $R^2$ es alto y espurio; use error fuera de muestra y MASE |
| Reportar el modelo final sin comparar contra una referencia trivial | Se asume que un ARIMA siempre aporta | Incluir siempre naive estacional y naive con deriva |
| Omitir el diagnóstico de homocedasticidad | Es parte del diagnóstico completo y suele olvidarse | ARCH-LM **y** Goldfeld–Quandt: miden cosas distintas |

---
# 11. Glosario español–inglés

| Español | Inglés | Símbolo / función |
|---|---|---|
| Operador de rezago | *backshift / lag operator* | $B$ |
| Diferencia regular | *regular differencing* | $\nabla = (1-B)$ · `.diff()` |
| Diferencia estacional | *seasonal differencing* | $\nabla_s = (1-B^s)$ · `.diff(s)` |
| Sobre-diferenciación | *overdifferencing* | — |
| Media móvil (modelo) | *moving average* | MA($q$) · $\theta$ |
| Autorregresivo | *autoregressive* | AR($p$) · $\phi$ |
| Invertibilidad | *invertibility* | raíces de $\theta(B)$ fuera del círculo unitario |
| Parsimonia | *parsimony* | — |
| Máxima verosimilitud | *maximum likelihood estimation* | MLE · `.fit()` |
| Criterio de información | *information criterion* | AIC · BIC · AICc |
| Modelo de aerolínea | *airline model* | $SARIMA(0,1,1)(0,1,1)_s$ |
| Fuera de muestra | *out-of-sample* | — |
| Origen móvil | *rolling origin / time series CV* | — |
| Intervalo de predicción | *prediction interval* | `get_forecast().conf_int()` |
| Ruido blanco | *white noise* | $\varepsilon_t$ |
| Heterocedasticidad | *heteroskedasticity* | ARCH-LM · Goldfeld–Quandt |
| Sesgo de retransformación | *retransformation bias* | $e^{\sigma^2/2}$ |
| Error escalado | *scaled error* | MASE |

---
# 12. Autoevaluación de cierre

Marque solo lo que pueda explicarle a un compañero **sin mirar el notebook**:

- ☐ Puedo escribir un $SARIMA(p,d,q)(P,D,Q)_s$ con el operador de rezago y decir qué representa cada polinomio.
- ☐ Puedo explicar por qué un ARIMA es un ARMA sobre la serie diferenciada.
- ☐ Puedo justificar el orden logaritmo → diferencia estacional → diferencia regular.
- ☐ Puedo detectar sobre-diferenciación sin usar ADF.
- ☐ Puedo distinguir el bloque regular del bloque estacional en una ACF y decir qué orden determina cada uno.
- ☐ Puedo enunciar las dos condiciones bajo las cuales dos AIC son comparables.
- ☐ Puedo explicar por qué `np.exp()` sobre un pronóstico logarítmico subestima la media.
- ☐ Puedo explicar qué es la validación de origen móvil y por qué es preferible a una partición única.
- ☐ Puedo corregir los grados de libertad de Ljung–Box y decir por qué importa.
- ☐ Puedo nombrar los cuatro supuestos sobre los residuos y la consecuencia práctica de que falle cada uno.
- ☐ Puedo explicar la diferencia entre lo que detecta ARCH-LM y lo que detecta Goldfeld–Quandt.
- ☐ Puedo argumentar por qué el volumen a comprometer no es el pronóstico puntual.

**Si quedaron casillas sin marcar**, vuelva a la sección correspondiente antes de la Unidad 3: los modelos de suavizamiento y los métodos de aprendizaje automático se compararán contra el SARIMA que acaba de construir.

---
# 13. Laboratorio — Evaluación Formativa 2

<div style="border-left:6px solid #b30338; padding:8px 16px;">

**Producto:** informe técnico en notebook ejecutable de modelos clásicos (ARIMA / SARIMA)  
**Modalidad:** individual · **Carácter:** formativo, 0% de ponderación  
**Serie de trabajo:** `consumo_electrico_litoral.csv` (caso transversal), **no** la serie de esta sesión  
**Nombre del archivo:** `mcdi505_s2inicialapellido.ipynb` (por ejemplo, `mcdi505_s2jperez.ipynb`)  
**Extensión:** 10 a 15 páginas equivalentes · **Referencias:** formato APA  
**Plazo:** hasta las 23:59 hrs. del jueves de la Semana 2

</div>

Esta evaluación reutiliza la rúbrica de la Sumativa 2 y no pondera en la calificación final. Su puntaje se usa exclusivamente para retroalimentarle **antes** de la evaluación sumativa. La retroalimentación individual, con comentarios por criterio, se entrega dentro de los 2 días siguientes al cierre.

### Estructura exigida del informe

```
1. Portada / encabezado
2. Introducción y contexto del dataset
3. Análisis de estacionariedad y transformaciones          → Criterio 1
4. Identificación y estimación de modelos                  → Criterio 2
5. Selección y comparación de modelos                      → Criterio 3
6. Diagnóstico de residuos y validación de supuestos       → Criterio 4
7. Conclusiones
8. Referencias (APA)
9. Declaración de uso de asistentes de IA
```

### Reglas de trabajo

- El notebook debe ejecutarse **completo y sin errores desde un kernel reiniciado** (*Kernel → Restart & Run All*). Un notebook que no corre se evalúa como no logrado en el criterio de formato.
- **Rutas relativas**, nunca absolutas. `pd.read_csv("datos/consumo_electrico_litoral.csv")`, no `C:/Users/...`.
- Cada decisión técnica se documenta en el formato **decisión / evidencia / alternativa descartada / consecuencia**. Código sin interpretación escrita no se considera una respuesta completa.
- No modifique el archivo de datos original: toda limpieza se hace en código, dentro del notebook, para que sea auditable.
- **Uso de IA.** Está permitido usar asistentes para organizar ideas o mejorar la redacción, y **no** para producir respuestas completas. Debe declararlo explícitamente en la sección 9 y ser capaz de justificar por escrito cualquier decisión del notebook.

### Lista de cotejo — alineada con la rúbrica

| Criterio | Nivel *Excelente* (9 pts) exige… | ☐ |
|---|---|:--:|
| **1. Estacionariedad y transformaciones** | Aplicar **e interpretar** ADF y KPSS, decidir de forma justificada la transformación estabilizadora y los órdenes $d$ y $D$, y **verificar** la estacionariedad resultante. | ☐ |
| **2. Identificación y estimación** | Proponer candidatos **coherentes con la evidencia de ACF/PACF** y con la naturaleza estacional de la serie, y estimarlos correctamente. | ☐ |
| **3. Selección y comparación** | Comparar con criterios de información **y** métricas de ajuste o pronóstico, y seleccionar justificando con la evidencia cuantitativa. | ☐ |
| **4. Diagnóstico de residuos** | Diagnóstico **completo**: Ljung-Box, normalidad **y homocedasticidad**, con interpretación y conclusión sobre la validez del modelo. | ☐ |
| **5. Ortografía y redacción** | Redacción académica clara, precisa y cohesionada, sin errores relevantes. | ☐ |
| **6. Formato y entrega** | Notebook ejecutable, 10-15 páginas equivalentes, nombre de archivo correcto, referencias APA, entrega en plazo. | ☐ |

> ### ⚠️ Diferencias entre la serie de la sesión y la del proyecto
> No traslade las conclusiones de este notebook: trasládese el **método**. La serie de DEL tiene tres complicaciones que aquí no aparecen y que la rúbrica espera ver tratadas:
>
> 1. **Tres valores faltantes.** ADF, KPSS y la estimación MLE se comportan de manera distinta ante `NaN`. Decida e implemente una imputación *antes* de diferenciar, y justifíquela.
> 2. **Un dato atípico** (noviembre de 2017, recalibración de medidores). Un atípico infla $\hat\sigma^2$, ensancha los intervalos y puede hacer rechazar la normalidad.
> 3. **Un quiebre exógeno prolongado** (COVID-19, 2020-2021). La diferencia estacional compara cada mes con el mismo mes del año anterior: el shock contaminará observaciones de $w_t$ **dos veces**, al entrar y al salir. Cuantifique cuántas y decida qué hace con ellas.
>
> Además, su estacionalidad es **bimodal** —refrigeración en verano austral y calefacción en invierno—, de modo que la lectura de la ACF estacional no será tan limpia como la de esta sesión.

In [ ]:
# =============================================================================
# EJERCICIO 1 — Estacionariedad y transformaciones          (Criterio 1)
# Serie: consumo_gwh del caso transversal.
# =============================================================================

# TODO 1.1 — Cargue la serie con índice mensual declarado y resuelva los valores
#            faltantes. Justifique la estrategia de imputación en función del mes
#            en que cae cada hueco. Deje constancia de cuántos valores imputó.


# TODO 1.2 — Determine si la estacionalidad es aditiva o multiplicativa.
#            Use al menos DOS evidencias: el panel de media y desviación móviles
#            y el coeficiente de variación por año (§4.3 de esta sesión).


# TODO 1.3 — Decida la transformación estabilizadora. Estime λ de Box-Cox y
#            compare con el logaritmo. Justifique su elección.


# TODO 1.4 — Construya la escalera de diferenciación y determine d y D aplicando
#            ADF, KPSS y el CRITERIO DE VARIANZA. Incluya al menos un peldaño
#            sobre-diferenciado para mostrar que sabe reconocerlo.


# TODO 1.5 — Verifique la estacionariedad de la serie resultante y comente el
#            efecto del quiebre COVID-19 sobre las pruebas.


# --- Respuesta escrita (formato exigido) ---
# Decisión:
# Evidencia:
# Alternativa descartada:
# Consecuencia para las secciones siguientes:

In [ ]:
# =============================================================================
# EJERCICIO 2 — Identificación y estimación                 (Criterio 2)
# =============================================================================

# TODO 2.1 — Grafique ACF y PACF de la serie estacionaria, marcando los rezagos
#            estacionales. Reporte la banda ±1.96/√n y qué rezagos la superan.


# TODO 2.2 — Argumente la lectura por bloques: qué sugiere el bloque regular
#            (rezagos 1 a 11) para p y q, y qué sugiere el bloque estacional
#            (12, 24, 36) para P y Q. Atención a la estacionalidad bimodal.


# TODO 2.3 — Proponga AL MENOS CUATRO especificaciones candidatas, cada una con
#            una frase que explique por qué está en la lista. Incluya
#            deliberadamente una lectura alternativa a la suya.


# TODO 2.4 — Defina la partición cronológica y JUSTIFIQUE el tamaño del conjunto
#            de prueba en relación con el horizonte de decisión del caso.


# TODO 2.5 — Construya al menos dos modelos de referencia (naive estacional y
#            naive estacional con deriva) y calcule sus métricas. Esta es la vara.


# TODO 2.6 — Estime los candidatos por máxima verosimilitud. Verifique
#            convergencia, invertibilidad (módulo de las raíces) y significancia
#            de los coeficientes. Descarte los que no cumplan ANTES de comparar.


# --- Respuesta escrita ---
# Órdenes candidatos y su fundamento en la evidencia:
# Partición elegida y justificación:

In [ ]:
# =============================================================================
# EJERCICIO 3 — Selección y comparación                     (Criterio 3)
# =============================================================================

# TODO 3.1 — Tabla comparativa con AIC, BIC y AICc. Declare explícitamente qué
#            modelos son comparables entre sí y por qué (§6.1).


# TODO 3.2 — Pronostique sobre el conjunto de prueba en la ESCALA ORIGINAL,
#            aplicando la corrección del sesgo de retransformación si trabajó en
#            logaritmos. Reporte RMSE, MAE, MAPE y MASE.


# TODO 3.3 — Ejecute una validación de origen móvil con al menos 8 orígenes.
#            ¿Coincide el ranking con el de la partición única? Interprete.


# TODO 3.4 — Compare contra los modelos de referencia del ejercicio 2.
#            ¿Su modelo APORTA? Responda con la cifra, no con una impresión.


# TODO 3.5 — Seleccione el modelo final. Si elige uno con AIC ligeramente peor
#            pero más parsimonioso, argumente esa decisión explícitamente.


# --- Respuesta escrita ---
# Decisión:
# Evidencia cuantitativa:
# Alternativa descartada:
# Consecuencia:

In [ ]:
# =============================================================================
# EJERCICIO 4 — Diagnóstico de residuos                     (Criterio 4)
# =============================================================================

# TODO 4.1 — Obtenga los residuos del modelo seleccionado DESCARTANDO las
#            primeras d + D·s observaciones. Muestre el efecto de no hacerlo.


# TODO 4.2 — Panel gráfico de cuatro cuadrantes: residuos en el tiempo,
#            histograma con densidad normal, gráfico Q-Q y ACF de los residuos.


# TODO 4.3 — Ljung-Box en al menos tres valores de K, CON los grados de libertad
#            corregidos por el número de parámetros ARMA estimados.


# TODO 4.4 — Normalidad: Shapiro-Wilk y Jarque-Bera. Reporte asimetría y curtosis.


# TODO 4.5 — Homocedasticidad: ARCH-LM Y Goldfeld-Quandt. Explique qué mide cada
#            una y por qué pueden discrepar.


# TODO 4.6 — Concluya sobre la VALIDEZ del modelo. Si algún supuesto falla,
#            indique si es bloqueante, cuál es la causa probable y qué efecto
#            tiene sobre el pronóstico y sobre los intervalos.


# --- Respuesta escrita ---
# Supuestos que se cumplen:
# Supuestos que no se cumplen y por qué:
# ¿El modelo es utilizable? Justifique:
# Limitaciones que declararía al cliente:

In [ ]:
# =============================================================================
# EJERCICIO 5 (DESAFÍO, opcional) — El costo de una decisión mal fundamentada
# =============================================================================
# Este ejercicio no suma puntaje, pero anticipa la Semana 4.
#
# TODO 5.1 — Ajuste el mismo modelo DOS veces: una tratando el periodo COVID-19
#            con variables indicadoras y otra ignorándolo por completo.
#
# TODO 5.2 — Compare los pronósticos de 2024 de ambas versiones. ¿Cuánta energía
#            de diferencia hay entre ambos, mes a mes y en el total anual?
#
# TODO 5.3 — Traduzca esa diferencia a la decisión del caso: si DEL contratara
#            según la versión que ignora el shock, ¿estaría contratando de más o
#            de menos, y en qué magnitud?

### Declaración de uso de asistentes de IA

*(Complete esta sección en su entrega. Es obligatoria y su ausencia afecta el criterio de formato.)*

| Pregunta | Respuesta |
|---|---|
| ¿Utilizó asistentes de IA en esta entrega? | |
| ¿En qué partes específicas? | |
| ¿Qué verificó de forma independiente y cómo? | |

---

## Próxima sesión

**Sesión 3 — Suavizamiento exponencial y modelos de espacio de estados.** Construiremos la familia ETS (*error, trend, seasonal*), veremos en qué se parece y en qué difiere de un SARIMA, y compararemos ambas familias sobre la misma serie usando el protocolo de validación de esta sesión. El SARIMA que acaba de construir pasa a ser **la referencia contra la cual se evalúan los métodos siguientes**.

**Preparación sugerida:** revisar los capítulos 8 y 9 de Hyndman y Athanasopoulos (2021) y traer resuelto el Ejercicio 1 del laboratorio, que es insumo directo de la Semana 3.

---

# 14. Referencias

Box, G. E. P., Jenkins, G. M., Reinsel, G. C., & Ljung, G. M. (2016). *Time series analysis: Forecasting and control* (5th ed.). Wiley.

Burnham, K. P., & Anderson, D. R. (2004). Multimodel inference: Understanding AIC and BIC in model selection. *Sociological Methods & Research, 33*(2), 261–304. https://doi.org/10.1177/0049124104268644

Bianchi, F. M. (2023). *Time series analysis with Python* [Repositorio de GitHub]. https://github.com/FilippoMB/python-time-series-handbook

Hyndman, R. J., & Athanasopoulos, G. (2021). *Forecasting: Principles and practice* (3rd ed.). OTexts. https://otexts.com/fpp3/

Hyndman, R. J., & Koehler, A. B. (2006). Another look at measures of forecast accuracy. *International Journal of Forecasting, 22*(4), 679–688. https://doi.org/10.1016/j.ijforecast.2006.03.001

Kulkarni, A. R., Shivananda, A., Kulkarni, A., & Krishnan, V. A. (2023). *Time series algorithms recipes: Implement machine learning and deep learning techniques with Python*. Apress. https://doi.org/10.1007/978-1-4842-8978-5

Kwiatkowski, D., Phillips, P. C. B., Schmidt, P., & Shin, Y. (1992). Testing the null hypothesis of stationarity against the alternative of a unit root. *Journal of Econometrics, 54*(1–3), 159–178. https://doi.org/10.1016/0304-4076(92)90104-Y

Ljung, G. M., & Box, G. E. P. (1978). On a measure of lack of fit in time series models. *Biometrika, 65*(2), 297–303. https://doi.org/10.1093/biomet/65.2.297

Seabold, S., & Perktold, J. (2010). Statsmodels: Econometric and statistical modeling with Python. *Proceedings of the 9th Python in Science Conference*, 92–96. https://doi.org/10.25080/Majora-92bf1922-011

Tashman, L. J. (2000). Out-of-sample tests of forecasting accuracy: An analysis and review. *International Journal of Forecasting, 16*(4), 437–450. https://doi.org/10.1016/S0169-2070(00)00065-0

Woodward, W. A., Sadler, B. P., & Robertson, S. (2022). *Time series for data science: Analysis and forecasting*. CRC Press. https://doi.org/10.1201/9781003089070

---

<div style="text-align:center; color:#777; font-size:0.85em;">
MCDI505 — Análisis de Series de Tiempo · Unidad 2 · Magíster en Ciencia de Datos · UNAB Online
</div>